# SVD 与特征值分解：从朴素算法到工业级实现

在深入现代工业级极速 SVD（奇异值分解）与特征值求解算法之前，我们首先需要理解其背后的底层基石——**朴素算法**。接下来的几个代码块展示了三个核心的基础算法实现，它们是所有复杂优化的理论起点：

### 1. 朴素 Householder 双对角化 (Householder Bidiagonalization)
要对一个庞大的稠密矩阵直接求 SVD 是极其困难的。第一步通常是利用 Householder 变换（空间反射），将任意稠密矩阵 $A$ 转化为一个**双对角矩阵**。

* **什么是 Householder 反射？** Householder 反射是一种正交变换，它能够将空间中的一个向量沿某个超平面进行“镜像反射”。在数值计算中，它的超级能力在于：可以精心构造一个反射面，把一个向量的特定分量（比如对角线以下的所有元素）一次性“反射”成 0，同时绝对保持向量的模长不变（保证了正交性）。
* **什么是双对角矩阵？** 双对角矩阵是指除了主对角线和第一超对角线（主对角线正上方的一条斜线）之外，其他所有元素全为 0 的矩阵。
* **前提要求**：该算法通常要求输入矩阵的行数大于等于列数（**$m \ge n$**）。如果 $m < n$，我们可以先对矩阵转置后再处理。
* **具体过程**：
  1. **左侧变换（消减列）**：对于第 $k$ 列，构造一个 Householder 反射矩阵并左乘矩阵，将该列主对角线以下的所有元素全部消为 0。
  2. **右侧变换（消减行）**：为了不破坏刚刚消好的列，紧接着在第 $k$ 行构造另一个 Householder 矩阵并右乘，将该行第一超对角线右侧的所有元素消为 0。
  3. 交替进行上述左右操作，从左上角一路推进到右下角，最终将整个矩阵“削减”为双对角形态。
* **核心意义**：这是降维打击的第一步，把 $O(m \times n)$ 的全局消元问题，压缩成了只需关注主对角线和超对角线的局部问题。

### 2. 朴素 QR 分解 (Modified Gram-Schmidt)
QR 分解是迭代求特征值的引擎。在这部分实现中，我们采用了**改进的格拉姆-施密特正交化 (Modified Gram-Schmidt, MGS)**。
* **什么是 QR 分解？** QR 分解是指将任意矩阵 $A$ 分解为一个列正交矩阵 $Q$（满足 $Q^T Q = I$，它的列向量构成了空间的一组标准正交基）和一个上三角矩阵 $R$（记录了原向量在这些正交基上的投影系数）的乘积，即 $A = QR$。
* **核心意义**：与课本上标准的施密特正交化相比，MGS 在每次求出一个正交基后，会立刻从后续**所有列**中减去该方向的投影。这种运算顺序的微调，极大地抵抗了计算机浮点数运算带来的舍入误差，保证了矩阵 $Q$ 的正交性。

### 3. 朴素 QR 迭代 (Naive QR Iteration)
这是求解**对称方阵**特征值的最基础方法：不断地对矩阵进行 $A = QR$ 分解，然后再反向相乘得到 $A_{next} = RQ$。随着迭代进行，矩阵会逐渐趋于上三角阵，对角线上的元素即为特征值。
* **核心意义与局限**：它揭示了“QR 迭代等价于多维幂法”的数学美感。但由于每次完整的 QR 分解都需要 $O(n^3)$ 的复杂度，并且面对相近特征值时收敛极其缓慢，这种朴素方法在工业界是不可接受的。关于为什么QR迭代最终会完成SVD分解，之后再讨论（甚至实际上，朴素的QR迭代不一定会收敛）。

* **🚨 重要概念澄清：理论桥梁与不可直接串联**
  必须指出，**我们不能直接对第一步算出的双对角矩阵 $B$ 执行此处的 QR 迭代！** 前面的 Householder 双对角化是为了求 SVD，而这里的 QR 迭代针对的是**对称方阵**的特征值求解。
  SVD 和特征值分解的理论桥梁在于：矩阵的奇异值，等于对称矩阵 $A^T A$ (或 $B^T B$) 特征值的平方根。因此，朴素的 SVD 逻辑应该是：先双对角化得到 $B$，构造出对称三对角矩阵 $T = B^T B$，**再对对称方阵 $T$ 进行 QR 迭代**。这也解释了为什么在随后的测试代码中，我们没有直接传入双对角矩阵 $B$，而是专门构造了一个新的对称方阵 `A_square = M.T @ M` 来演示 QR 迭代的正确用法。至于为什么工业界**绝对不能**显式地去计算 $B^T B$，这正是后续我们要揭秘的隐式算法的核心！

理解了这些朴素算法的痛点，我们就能彻底明白后续引入 **O(n) Givens 旋转**、**Wilkinson Shift (原点平移)** 和 **追赶鼓包 (Bulge Chasing)** 等工业级黑科技的真正动机。

In [1]:
import numpy as np

def householder_bidiagonalization(A):
    """
    使用Householder变换将 m x n (m >= n) 矩阵A转化为双对角矩阵。

    返回:
        B: 双对角矩阵
        U_vecs: 左侧Householder向量序列
        V_vecs: 右侧Householder向量序列
    """
    m, n = A.shape
    if m < n:
        raise ValueError("需要满足 m >= n")

    B = A.copy().astype(float)
    U_vecs = []
    V_vecs = []

    for k in range(n):
        # 1. 消除第k列对角线以下的元素 (左侧Householder变换)
        x = B[k:m, k]
        if len(x) > 1:
            e1 = np.zeros_like(x)
            e1[0] = 1.0

            sign_x = np.sign(x[0]) if x[0] != 0 else 1.0
            v_left = x + sign_x * np.linalg.norm(x) * e1

            norm_v_left = np.linalg.norm(v_left)
            if norm_v_left > 1e-15:
                v_left = v_left / norm_v_left
                U_vecs.append((k, v_left.copy()))

                # 更新B矩阵
                B[k:m, k:n] -= 2.0 * np.outer(v_left, np.dot(v_left.T, B[k:m, k:n]))

        # 2. 消除第k行超对角线右侧的元素 (右侧Householder变换)
        if k < n - 2:
            y = B[k, k+1:n]
            if len(y) > 1:
                e1_r = np.zeros_like(y)
                e1_r[0] = 1.0

                sign_y = np.sign(y[0]) if y[0] != 0 else 1.0
                v_right = y + sign_y * np.linalg.norm(y) * e1_r

                norm_v_right = np.linalg.norm(v_right)
                if norm_v_right > 1e-15:
                    v_right = v_right / norm_v_right
                    V_vecs.append((k, v_right.copy()))

                    # 更新B矩阵
                    B[k:m, k+1:n] -= 2.0 * np.outer(np.dot(B[k:m, k+1:n], v_right), v_right.T)

    # 为了使结果严格为双对角形式，清理由于浮点数误差产生的微小非零项
    B = np.triu(np.tril(B, 1))

    return B, U_vecs, V_vecs

# 测试代码
np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
A = np.random.rand(5, 4) * 10
B, U_vecs, V_vecs = householder_bidiagonalization(A)

print("原始矩阵 A:")
print(A)
print("\n双对角化后的矩阵 B:")
print(B)
print(f"\n收集到的左侧Householder向量数量: {len(U_vecs)}")
print(f"收集到的右侧Householder向量数量: {len(V_vecs)}")

原始矩阵 A:
[[3.7454 9.5071 7.3199 5.9866]
 [1.5602 1.5599 0.5808 8.6618]
 [6.0112 7.0807 0.2058 9.6991]
 [8.3244 2.1234 1.8182 1.834 ]
 [3.0424 5.2476 4.3195 2.9123]]

双对角化后的矩阵 B:
[[-11.452   15.2269   0.       0.    ]
 [  0.      12.8505   0.5616   0.    ]
 [  0.       0.      -3.4267   4.6465]
 [  0.       0.       0.      -6.0882]
 [  0.       0.       0.       0.    ]]

收集到的左侧Householder向量数量: 4
收集到的右侧Householder向量数量: 2


In [2]:
import numpy as np

def naive_qr_decomposition(A):
    """
    使用改进的格拉姆-施密特正交化(Modified Gram-Schmidt)实现朴素的QR分解(经济型)。
    输入: A (m x n 矩阵, m >= n)
    输出:
        Q (m x n 矩阵): 列正交矩阵 (Q.T @ Q = I)
        R (n x n 矩阵): 上三角矩阵
    """
    m, n = A.shape
    if m < n:
        raise ValueError("需要满足 m >= n")

    Q = np.zeros((m, n))
    R = np.zeros((n, n))

    # 复制A以免修改原矩阵，通常用V表示正在正交化的列
    V = A.copy().astype(float)

    for i in range(n):
        # 1. 计算当前列的模长，作为 R 的对角元素
        v = V[:, i]
        R[i, i] = np.linalg.norm(v)

        # 如果范数接近0，说明列线性相关（矩阵非满秩）
        if R[i, i] > 1e-15:
            Q[:, i] = v / R[i, i]
        else:
            Q[:, i] = np.zeros(m) # 发生秩亏缺时的简单处理

        # 2. 从后面的所有列中减去当前正交基的投影
        # 这是 Modified Gram-Schmidt 的核心，比每次都用原始A进行投影更稳定
        for j in range(i + 1, n):
            R[i, j] = np.dot(Q[:, i], V[:, j])
            V[:, j] = V[:, j] - R[i, j] * Q[:, i]

    return Q, R

# 测试代码
np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
A_test = np.random.rand(5, 3) * 10
Q, R = naive_qr_decomposition(A_test)

print("原始矩阵 A:")
print(A_test)
print("\n正交矩阵 Q (m x n):")
print(Q)
print("\n验证 Q 矩阵的正交性 (Q.T @ Q 应该接近单位矩阵):")
print(Q.T @ Q)
print("\n上三角矩阵 R (n x n):")
print(R)
print("\n重建的 A (Q @ R):")
print(Q @ R)


原始矩阵 A:
[[3.7454 9.5071 7.3199]
 [5.9866 1.5602 1.5599]
 [0.5808 8.6618 6.0112]
 [7.0807 0.2058 9.6991]
 [8.3244 2.1234 1.8182]]

正交矩阵 Q (m x n):
[[ 0.2876  0.6645  0.0252]
 [ 0.4596 -0.0732 -0.3145]
 [ 0.0446  0.7015  0.1278]
 [ 0.5436 -0.2231  0.8078]
 [ 0.6391 -0.1056 -0.4812]]

验证 Q 矩阵的正交性 (Q.T @ Q 应该接近单位矩阵):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

上三角矩阵 R (n x n):
[[13.0245  5.3064  9.525 ]
 [ 0.     12.01    6.6117]
 [ 0.      0.      7.422 ]]

重建的 A (Q @ R):
[[3.7454 9.5071 7.3199]
 [5.9866 1.5602 1.5599]
 [0.5808 8.6618 6.0112]
 [7.0807 0.2058 9.6991]
 [8.3244 2.1234 1.8182]]


In [3]:
def naive_qr_iteration(A, num_iter=200, tol=1e-10):
    """
    使用朴素的QR迭代计算矩阵的特征值及对应的正交变换矩阵。
    输入: A (n x n 方阵)
    """
    n, m = A.shape
    if n != m:
        raise ValueError("QR迭代求特征值需要输入方阵")

    A_k = A.copy().astype(float)
    # 初始化累积正交矩阵为单位阵
    Q_acc = np.eye(n)

    for k in range(num_iter):
        # 1. 调用前面实现的朴素QR分解
        Q, R = naive_qr_decomposition(A_k)

        # 2. 累积正交矩阵 (Q_total = Q_0 @ Q_1 @ ... @ Q_k)
        Q_acc = Q_acc @ Q

        # 3. 核心步骤: 反向相乘更新 A
        A_next = R @ Q

        # 4. 检查收敛性：看次对角线以下元素是否足够接近0
        off_diag_max = np.max(np.abs(np.tril(A_next, -1)))
        if off_diag_max < tol:
            print(f"在第 {k+1} 次迭代后收敛。")
            A_k = A_next
            break

        A_k = A_next

    # 收敛后，对角线上的元素即为特征值
    eigenvalues = np.diag(A_k)
    return eigenvalues, A_k, Q_acc

# 测试代码
np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

# 构造一个对称矩阵(保证特征值为实数且容易收敛)
n = 4
M = np.random.rand(n, n) * 5
A_square = M.T @ M

print("原始对称方阵 A:")
print(A_square)

eigvals, A_inf, Q_acc = naive_qr_iteration(A_square, num_iter=500, tol=1e-12)

print("\n收敛后的矩阵 (接近对角矩阵):")
print(A_inf)

print("\n计算得到的特征值:")
print(eigvals)

print("\n累积的正交矩阵 Q_acc (对于对称阵，即为特征向量矩阵):")
print(Q_acc)

print("\n验证分解 A = Q_acc @ A_inf @ Q_acc.T:")
print(Q_acc @ A_inf @ Q_acc.T)


原始对称方阵 A:
[[30.4731 24.5703 11.1739 27.3766]
 [24.5703 36.8662 18.954  35.7496]
 [11.1739 18.954  14.3168 13.5459]
 [27.3766 35.7496 13.5459 52.0754]]
在第 149 次迭代后收敛。

收敛后的矩阵 (接近对角矩阵):
[[107.1754   0.      -0.      -0.    ]
 [ -0.      13.7418   0.       0.    ]
 [ -0.       0.      11.1322   0.    ]
 [  0.       0.      -0.       1.682 ]]

计算得到的特征值:
[107.1754  13.7418  11.1322   1.682 ]

累积的正交矩阵 Q_acc (对于对称阵，即为特征向量矩阵):
[[ 0.448   0.1999  0.8689  0.0656]
 [ 0.5569  0.3543 -0.3172 -0.6809]
 [ 0.2622  0.6029 -0.3252  0.6797]
 [ 0.6484 -0.6863 -0.1964  0.2647]]

验证分解 A = Q_acc @ A_inf @ Q_acc.T:
[[30.4731 24.5703 11.1739 27.3766]
 [24.5703 36.8662 18.954  35.7496]
 [11.1739 18.954  14.3168 13.5459]
 [27.3766 35.7496 13.5459 52.0754]]


### 深入理解朴素QR迭代的收敛性与迭代次数

在我们上面实现的“朴素QR迭代”中，你可能会发现有时只需几十次就能收敛，有时却需要成百上千次。理解这背后的原因，是迈向工业级特征值/SVD算法（如LAPACK中的实现）的关键。

#### 1. 决定迭代次数的数学本质：特征值的比值与“次对角线”瓶颈
在我们的代码中，输入的是一个稠密对称方阵，随着迭代进行，整个“严格下三角区域”都在逐渐趋于0。但我们在理论分析时，通常会特别盯住**次对角线（sub-diagonal，$a_{i, i-1}$）**。为什么呢？

因为次对角线是整个下三角阵中**收敛最慢、最“顽固”的部分**。

朴素QR迭代的收敛速度是**线性**的。假设矩阵 $A$ 的特征值按绝对值大小降序排列为：
$$|\lambda_1| > |\lambda_2| > \dots > |\lambda_n|$$

对于一般的稠密矩阵，经过 $k$ 次迭代后，任意下三角元素 $a_{i, j}^{(k)}$（其中 $i > j$）趋于0的速度取决于对应特征值的比值：
$$ |a_{i, j}^{(k)}| \sim O\left( \left| \frac{\lambda_i}{\lambda_j} \right|^k \right) $$

* **跨度大的元素（如左下角）收敛极快**：比如 $a_{3, 1}$ 衰减取决于 $\lambda_3 / \lambda_1$。因为这两个特征值隔得远，比值通常很小，所以会极快地变成 0。
* **相邻的元素（即次对角线）收敛最慢**：比如 $a_{2, 1}$ 衰减取决于相邻特征值的比值 $\lambda_2 / \lambda_1$。这是所有组合中最接近 1 的比值。因此，次对角线上的元素是下三角区域最后变为 0 的地方，决定了整个算法的迭代次数。

#### 2. 为什么收敛速度是特征值的比值？（数学证明简述）
要证明这个衰减率，我们需要利用QR迭代的一个极具美感的性质：**QR迭代实际上是多维的幂法（Simultaneous Subspace Iteration）**。

让我们定义累积的正交矩阵和上三角矩阵：
$$ \underline{Q}_k = Q_1 Q_2 \dots Q_k $$
$$ \underline{R}_k = R_k R_{k-1} \dots R_1 $$

**核心定理**：进行 $k$ 次QR迭代所得到的 $\underline{Q}_k$ 和 $\underline{R}_k$，恰好是对矩阵 $A$ 的 $k$ 次幂 $A^k$ 直接进行QR分解的结果，即：
$$ A^k = \underline{Q}_k \underline{R}_k $$

**核心定理的证明（数学归纳法）**：
首先，由QR迭代的定义可知 $A_{i} = Q_i R_i$，且 $A_{i+1} = R_i Q_i$。因此 $A_{i+1} = Q_i^T A_i Q_i$。
递归展开可得：$A_{k+1} = (Q_1 \dots Q_k)^T A_1 (Q_1 \dots Q_k) = \underline{Q}_k^T A \underline{Q}_k$。
移项可得一个重要引理：**$A \underline{Q}_k = \underline{Q}_k A_{k+1}$**。

现在我们用归纳法证明 $A^k = \underline{Q}_k \underline{R}_k$：
* **当 $k=1$ 时**：$A^1 = Q_1 R_1 = \underline{Q}_1 \underline{R}_1$，显然成立。
* **假设 $k-1$ 时成立**：即 $A^{k-1} = \underline{Q}_{k-1} \underline{R}_{k-1}$。
* **当为 $k$ 时**：
  $$ A^k = A \cdot A^{k-1} = A (\underline{Q}_{k-1} \underline{R}_{k-1}) = (A \underline{Q}_{k-1}) \underline{R}_{k-1} $$
  利用上面的引理 $A \underline{Q}_{k-1} = \underline{Q}_{k-1} A_k$，代入得：
  $$ A^k = (\underline{Q}_{k-1} A_k) \underline{R}_{k-1} $$
  再利用 $A_k = Q_k R_k$ 代入：
  $$ A^k = \underline{Q}_{k-1} (Q_k R_k) \underline{R}_{k-1} = (\underline{Q}_{k-1} Q_k) (R_k \underline{R}_{k-1}) $$
  根据定义，这正好等于 $\underline{Q}_k \underline{R}_k$。证明完毕！

**收敛速率推导（从定理到收敛率）**：

既然我们证明了 $A^k = \underline{Q}_k \underline{R}_k$，我们需要利用上三角矩阵 $\underline{R}_k$ 的一个重要特性：**在矩阵乘法中，乘以一个上三角矩阵，意味着结果矩阵的第 $i$ 列仅仅是原矩阵前 $i$ 列的线性组合。**

换句话说，$A^k$ 的前 $i$ 列的线性张成空间，完全等价于 $\underline{Q}_k$ 的前 $i$ 列的线性张成空间。因此，**$\underline{Q}_k$ 的前 $i$ 列就是矩阵 $A^k$ 前 $i$ 列所张成空间的正交基**。

假设 $A$ 的特征分解为 $A = U \Lambda U^T$。那么 $A^k = U \Lambda^k U^T$。

1. **子空间迭代（幂法）的真正含义与前提**：
   矩阵 $A^k$ 的前 $i$ 列，其实质就是将算子 $A^k$ 作用于单位矩阵 $I$ 的前 $i$ 个标准基向量 $e_1, e_2, \dots, e_i$ 上。
   **这里有一个至关重要的数学前提假设（非退化假设）**：这前 $i$ 个初始标准基向量，在 $A$ 的前 $i$ 个最大特征值对应的特征向量空间上的**投影必须不为0**。如果它们完全正交，理论上幂法永远无法提纯出主特征空间的成分。
   只要满足这个前提，根据幂法的原理，任何初始向量用 $A$ 映射 $k$ 次后，其方向都会被主特征向量方向主导。
   具体到这前 $i$ 个列向量，随着 $k$ 的增大，它们张成的子空间会向 $A$ 的前 $i$ 个**绝对值最大的特征向量**所张成的主子空间“靠拢”。
   在靠拢的过程中，第 $i+1$ 个特征向量方向上的成分属于“杂质”。由于第 $i$ 个方向每次被放大 $\lambda_i$ 倍，而第 $i+1$ 个方向每次被放大 $\lambda_{i+1}$ 倍，所以在经过 $k$ 次映射后，杂质相对主成分的比例就是 $( \lambda_{i+1} / \lambda_i )^k$。
   *(注：在实际编程中，即使初始理论正交，极其微小的浮点数舍入误差也会瞬间在主子空间方向产生非零投影分量，并在随后的迭代中被迅速放大。但在严谨的数学证明中，必须依赖该前提假设)*。

2. **正交基的收敛**：
   因为 $\underline{Q}_k$ 的前 $i$ 列（记为 $q_1^{(k)}, \dots, q_i^{(k)}$）正是 $A^k$ 前 $i$ 列空间的正交基，所以随着 $k$ 增大，$\underline{Q}_k$ 的第 $i$ 列 $q_i^{(k)}$ 会收敛到真实的第 $i$ 个特征向量 $u_i$。它距离真正特征向量的距离（即包含的杂质分量）会以 $O(|\lambda_{i+1} / \lambda_i|^k)$ 的速率衰减。

3. **回到次对角线元素**：
   我们在第 $k$ 步得到的更新矩阵是 $A_{k+1} = \underline{Q}_k^T A \underline{Q}_k$。
   它的次对角线元素 $a_{i+1, i}^{(k)}$ 的计算公式，从矩阵乘法展开来看实际上是：
   $$ a_{i+1, i}^{(k)} = (q_{i+1}^{(k)})^T A q_i^{(k)} $$
   如果 $q_i^{(k)}$ 已经完美收敛到了真实的特征向量 $u_i$，那么 $A q_i^{(k)} = \lambda_i u_i$。
   此时代入上式得到 $\lambda_i (q_{i+1}^{(k)})^T u_i$。因为 $q_{i+1}^{(k)}$ 也会趋近于 $u_{i+1}$，而正交矩阵的特征向量相互垂直（内积为0），所以这个值本该等于 0。
   **这个值之所以目前还不为0，唯一的原因就是 $q_i^{(k)}$ 还没有完全收敛，里面还残留着尚未被“滤除”的杂质。**
   既然这个残留误差以 $|\lambda_{i+1} / \lambda_i|^k$ 的速度衰减，那么次对角线元素 $a_{i+1, i}^{(k)}$ 趋近于 0 的速度必然也就受制于这个相同的衰减率 $O(|\lambda_{i+1} / \lambda_i|^k)$。

#### 3. 为什么工业界绝不会直接使用“朴素QR迭代”？
除了遇到相近特征值会导致迭代次数激增外，朴素QR分解还有个致命缺点：**单次迭代的计算代价太高**。
对一个 $n \times n$ 的稠密矩阵做一次完整的QR分解需要 $O(n^3)$ 的复杂度。如果迭代 $K$ 次，总复杂度是 $O(K \cdot n^3)$，这在处理大矩阵时是不可接受的。

#### 4. 现代算法是如何减少迭代次数的？
既然最顽固的敌人是“次对角线”，而其他左下角元素反正也会极快归零，工业界的算法（如Francis QR Step）引入了三大杀器：

1. **矩阵预处理 (Hessenberg / Tridiagonal Reduction)**：
   在开始迭代前，先用Householder变换把一般稠密矩阵强行化为**上Hessenberg矩阵**，或者把对称矩阵化为**三对角矩阵**。一次性消灭掉非相邻的下三角元素。这不仅让算法专心对付次对角线，还使得每次QR分解的代价直接从 $O(n^3)$ 暴降到 $O(n^2)$ 甚至 $O(n)$（就像我们后续用Givens旋转做的那样）。

2. **原点平移 (Shifts)**：
   既然收敛取决于比值 $|\lambda_{i+1} / \lambda_i|$，如果我们每次迭代时，从矩阵中减去一个标量 $\mu I$（即 $A_k - \mu_k I = Q_k R_k$），那么收敛比值就变成了：
   $$ \left| \frac{\lambda_{i+1} - \mu_k}{\lambda_{i} - \mu_k} \right| $$
   如果我们让猜测的 $\mu_k$ 非常接近右下角的特征值 $\lambda_{n}$，这个比值就会趋近于 0，从而实现**二次收敛甚至三次收敛**。常见的策略有 **Wilkinson Shift** 和 **Rayleigh Quotient Shift**。加入了带Shift的算法，通常平均只需 2~3 次迭代就能求出一个特征值。

3. **降阶技术 (Deflation)**：
   当右下角的某个次对角线元素因为 Shift 策略迅速衰减为 0 后，意味着对应位置的特征值已经求出。此时我们可以直接把矩阵“切开”，将最后一行/列剔除，只对左上角剩下的 $(n-1) \times (n-1)$ 子矩阵继续进行迭代。矩阵规模越变越小，计算越来越快。

### 深入解析代码：O(n) 三对角矩阵 QR 分解 (Givens 旋转)

下面的 `tridiagonal_qr` 函数实现了针对对称三对角矩阵的极速 QR 分解。为了让你完全看懂里面的每一个变量和步骤，我们来逐行进行拆解。

#### 1. 函数输入 (描述初始状态)
*   **`d` (主对角线)**: 长度为 $n$ 的一维数组。代表初始矩阵 $A$ 的主对角线元素。
*   **`e` (次/超对角线)**: 长度为 $n-1$ 的一维数组。因为 $A$ 是对称三对角矩阵，它的上（超对角线）和下（次对角线）是一模一样的，所以只用一个 `e` 传入即可表示完整的 ede 结构。

#### 2. 初始化变量 (构建目标矩阵 $R$ 的骨架)
QR 分解的目标是把矩阵 $A$ 变成上三角矩阵 $R$。由于 $A$ 是三对角的，通过 Givens 旋转后，$R$ 最多只有三条对角线有非零值。我们抛弃二维数组，直接用三个一维数组来表示 $R$：
*   **`R_d`**: 将成为目标矩阵 $R$ 的**主对角线**。初始从 `d` 复制而来。
*   **`R_e1`**: 将成为目标矩阵 $R$ 的**第一超对角线**。初始从 `e` 复制而来。
*   **`R_e2`**: 将成为目标矩阵 $R$ 的**第二超对角线**。**极其重要**：初始时这里是全 0 的。但在旋转消元的过程中，会产生“非零注入 (Fill-in)”现象，使得这个原本为 0 的位置产生数值。
*   **`c`, `s`**: 分别用于存储每一步 Givens 旋转的 $\cos(\theta)$ 和 $\sin(\theta)$ 值。
*   **`sub_diag`**: 这是我们的**消灭目标**！它代表当前矩阵下半部分的“次对角线”。随着循环的进行，这里的元素会被一个个变成 0。

#### 3. 核心循环：逐个消灭次对角线元素
循环 `for k in range(n - 1)` 的含义是：我们要依次处理前 $n-1$ 列，把主对角线正下方的那个元素（也就是次对角线元素）变成 0。

**步骤 3.1：锁定目标与计算旋转角度**
*   `x = R_d[k]`：当前主对角线上的元素（我们要用它来消灭它下方的元素）。
*   `y = sub_diag[k]`：正下方的次对角线元素（将被消灭的敌人）。
*   `r = np.hypot(x, y)`：计算 $\sqrt{x^2 + y^2}$。这就是旋转消元后，新的主对角线元素应该变成的值。
*   `c[k], s[k]`：计算出旋转矩阵的 $\cos$ 和 $\sin$ 参数 (即 `x/r` 和 `y/r`)。

**步骤 3.2：将旋转作用于第 $k$ 行和第 $k+1$ 行 (只影响这相邻的两行！)**
真正的 Givens 旋转相当于用一个 $2 \times 2$ 的旋转矩阵去乘这两行。因为矩阵绝大部分地方都是 0，我们只需要更新有值的那几列：

*   **更新第 $k$ 列 (战场中心)**:
    *   `R_d[k] = r`：主对角线元素更新为算出的模长 `r`。
    *   `sub_diag[k] = 0.0`：目标次对角线元素被成功消灭，变成 0。
*   **更新第 $k+1$ 列 (右侧的第一列)**:
    *   这里原本上、下两个元素分别是第一超对角线的 `R_e1[k]` (记为 `top1`) 和 下一行主对角线的 `R_d[k+1]` (记为 `bot1`)。
    *   用 $2 \times 2$ 旋转矩阵作用于这两个元素：
        `新的 top1 = c * top1 + s * bot1`
        `新的 bot1 = -s * top1 + c * bot1`
    *   代码中分别把新值写回了对应的 `R_e1[k]` 和 `R_d[k+1]` 数组中。
*   **更新第 $k+2$ 列 (右侧的第二列：见证 Fill-in 的诞生)**:
    *   在倒数第二次迭代之前 (`k < n - 2`)，第 $k+2$ 列也有值需要跟着旋转。
    *   这里原本上面的元素是第 $k$ 行的第 $k+2$ 列，由于初始是三对角矩阵，这个位置**原本是 0**，所以 `top2 = 0.0`。
    *   下面对应的元素是第 $k+1$ 行的第 $k+2$ 列，它正好是原本的第一超对角线元素 `R_e1[k+1]`，记为 `bot2`。
    *   旋转后：
        `新的 top2 = c * 0.0 + s * bot2` **(注意！由于原本是0加上了一个值，现在变成非零了！这就是新长出来的第二超对角线 `R_e2[k]`)**
        `新的 bot2 = -s * 0.0 + c * bot2` (其实就是等比例缩小了原来的 `R_e1[k+1]`)。

通过这种精打细算的一维数组交替更新，我们不仅完成了严格的 QR 分解，还完全避免了去内存里构造巨大的 $n \times n$ 全零矩阵，将空间和时间复杂度双双压低到了真正的工业级极限 $O(n)$。

In [4]:
import numpy as np

def tridiagonal_qr(d, e):
    """
    对对称三对角矩阵进行 O(n) 的 QR 分解 (使用 Givens 旋转)。

    输入:
        d: 一维数组, 长度为 n (主对角线)
        e: 一维数组, 长度为 n-1 (次对角线/超对角线)

    输出:
        c, s: Givens旋转的 cos 和 sin 序列 (长度均为 n-1)
        R_d: R 的主对角线 (长度 n)
        R_e1: R 的第一超对角线 (长度 n-1)
        R_e2: R 的第二超对角线 (长度 n-2)
    """
    n = len(d)

    # 初始化 R 的三条对角线
    R_d = d.copy().astype(float)
    R_e1 = e.copy().astype(float)
    R_e2 = np.zeros(n - 2) if n > 2 else np.array([])

    # 存储 Givens 旋转参数
    c = np.zeros(n - 1)
    s = np.zeros(n - 1)

    # 这是一个滚动变量，用来模拟当前正在被消灭的次对角线元素
    sub_diag = e.copy().astype(float)

    for k in range(n - 1):
        # 目标: 使用 R_d[k] 消灭 sub_diag[k]
        x = R_d[k]
        y = sub_diag[k]

        # 1. 计算 Givens 旋转参数 (c, s)
        r = np.hypot(x, y)
        if r == 0:
            c[k], s[k] = 1.0, 0.0
        else:
            c[k] = x / r
            s[k] = y / r

        # 2. 将旋转应用到第 k 行和第 k+1 行

        # 第 k 列 (主对角线与次对角线)
        R_d[k] = r
        sub_diag[k] = 0.0 # 被成功消灭

        # 第 k+1 列 (第一超对角线与下一行的主对角线)
        top1 = R_e1[k]
        bot1 = R_d[k+1]
        R_e1[k]  =  c[k] * top1 + s[k] * bot1
        R_d[k+1] = -s[k] * top1 + c[k] * bot1

        # 第 k+2 列 (会产生 fill-in，形成第二超对角线)
        if k < n - 2:
            # 原始情况: top2 对应的 (k, k+2) 在三对角阵中初始是 0
            # bot2 对应的 (k+1, k+2) 是初始的 e[k+1]
            top2 = 0.0
            bot2 = R_e1[k+1]
            R_e2[k]   =  c[k] * top2 + s[k] * bot2
            R_e1[k+1] = -s[k] * top2 + c[k] * bot2

    return c, s, R_d, R_e1, R_e2


# ================= 测试与验证代码 =================
np.set_printoptions(precision=4, suppress=True)
n = 5

# 随机生成一个对称三对角矩阵的 d 和 e
d_test = np.random.rand(n) * 10
e_test = np.random.rand(n - 1) * 5

print(f"输入主对角线 d: {d_test}")
print(f"输入次对角线 e: {e_test}")

# O(n) QR 分解
c, s, R_d, R_e1, R_e2 = tridiagonal_qr(d_test, e_test)

print("\n===== 分解结果 (完全 O(n) 存储) =====")
print(f"Givens c: {c}")
print(f"Givens s: {s}")
print(f"R主对角线 R_d: {R_d}")
print(f"R第一超对角线 R_e1: {R_e1}")
print(f"R第二超对角线 R_e2: {R_e2}")

# ================= 还原为稠密矩阵验证正确性 =================
print("\n===== 稠密矩阵还原验证 =====")

# 还原原始稠密矩阵 A
A_dense = np.diag(d_test) + np.diag(e_test, 1) + np.diag(e_test, -1)

# 还原稠密上三角矩阵 R
R_dense = np.diag(R_d) + np.diag(R_e1, 1)
if n > 2:
    R_dense += np.diag(R_e2, 2)

# 还原稠密正交矩阵 Q (G_0.T @ G_1.T @ ... @ G_{n-2}.T)
Q_dense = np.eye(n)
for k in range(n - 1):
    G_T = np.eye(n)
    # G_k.T 中的 2x2 旋转块
    G_T[k, k] = c[k]
    G_T[k, k+1] = -s[k]
    G_T[k+1, k] = s[k]
    G_T[k+1, k+1] = c[k]
    Q_dense = Q_dense @ G_T

print("原始矩阵 A_dense:")
print(A_dense)
print("\n还原的 Q_dense @ R_dense:")
print(Q_dense @ R_dense)
print(f"\n最大误差: {np.max(np.abs(A_dense - Q_dense @ R_dense)):.4e}")

输入主对角线 d: [3.0424 5.2476 4.3195 2.9123 6.1185]
输入次对角线 e: [0.6975 1.4607 1.8318 2.2803]

===== 分解结果 (完全 O(n) 存储) =====
Givens c: [0.9747 0.9593 0.8981 0.6286]
Givens s: [0.2235 0.2826 0.4398 0.7778]
R主对角线 R_d: [3.1213 5.1697 4.1655 2.9319 2.2529]
R第一超对角线 R_e1: [1.8524 2.5863 2.8588 6.0461]
R第二超对角线 R_e2: [0.3264 0.5176 1.0028]

===== 稠密矩阵还原验证 =====
原始矩阵 A_dense:
[[3.0424 0.6975 0.     0.     0.    ]
 [0.6975 5.2476 1.4607 0.     0.    ]
 [0.     1.4607 4.3195 1.8318 0.    ]
 [0.     0.     1.8318 2.9123 2.2803]
 [0.     0.     0.     2.2803 6.1185]]

还原的 Q_dense @ R_dense:
[[ 3.0424  0.6975  0.      0.     -0.    ]
 [ 0.6975  5.2476  1.4607  0.     -0.    ]
 [ 0.      1.4607  4.3195  1.8318  0.    ]
 [ 0.      0.      1.8318  2.9123  2.2803]
 [ 0.      0.      0.      2.2803  6.1185]]

最大误差: 1.7764e-15


### O(n) 的 QR 迭代：数据流向与“无用即丢弃”的哲学

正如刚才讨论的，对称三对角矩阵分解出的 $Q$ 和 $R$，在进行反向相乘 $RQ$ 时，宏观上 $A_{new} = Q^T A Q$ 依然是一个对称三对角矩阵。

在代码实现中，我们在计算右乘 Givens 旋转时，直接抛弃了第二超对角线 `e2[k-1]`。我们曾试图从纯代数角度去严格证明它在每一步计算出来恰好是 0。但如果我们换一个视角，从**算法的数据流向（Data Flow）和依赖关系**来看，其实存在一个更具计算思维、也更本质的解释：**我们根本不需要关心它是不是 0，只需要知道它再也没有用了！**

#### 为什么可以安全地丢弃第二超对角线？

让我们来看看计算过程中的信息是如何流动的：
我们在进行 $RQ$ 相乘时，是依次向右乘 $G_0, G_1, \dots, G_{n-2}$。
- 第 $k$ 步右乘 $G_k$，只会混合并更新矩阵的**第 $k$ 列和第 $k+1$ 列**。
- 一旦第 $k$ 步完成，进入第 $k+1$ 步（混合 $k+1$ 和 $k+2$ 列），**第 $k$ 列就彻底“定稿”了，此后无论怎么旋转，再也没有任何操作会触碰前 $k$ 列。**

现在来看我们在第 $k$ 步骤中算出的那个处于 $(k-1, k+1)$ 位置的元素，也就是代码里的 `e2[k-1]`（新的第二超对角线）：
1. **它所在的行是第 $k-1$ 行**。
2. **它会在下一步 $G_{k+1}$ 中参与计算吗？** 会的。它会和 $(k-1, k+2)$ 混合，生成新的 $(k-1, k+1)$ 和 $(k-1, k+2)$。并且这个“信息”会顺着第 $k-1$ 行，一直向右流动到最后一列。
3. **但是，这个信息流对最终结果有影响吗？** **完全没有！**
   因为最终我们只需要提取对称三对角矩阵的**主对角线 `d`** 和 **次对角线 `sub`**。
   对于第 $k-1$ 行，我们要提取的是主对角线元素 $(k-1, k-1)$；对于次对角线，我们要提取的是 $(k, k-1)$。
   注意它们的列索引：全都是 $k-1$ 列！
   而根据我们前面的分析，**第 $k-1$ 列在第 $k-1$ 步之后就已经彻底锁死了**。$(k-1, k-1)$ 这个格子的值早就定稿了。
   
这意味着，$(k-1, k+1)$ 这个位置的元素，无论它算出来是 0 还是别的什么数字，它在未来的所有旋转中，都**只会在第 $k-1$ 行的更右侧打转**（影响 $(k-1, k+2)$, $(k-1, k+3)$ 等等）。它**永远、永远不可能再向左或向下“拐弯”影响到任何主对角线或次对角线上的元素**（因为那些列和行早就脱离了它的作用域）！

**结论：**
宏观的数学定理告诉我们，最终矩阵是对称三对角的，所以这个向右流动的信息最终必定会收敛归零。
但从计算机科学的角度，我们得出了一个极其霸道的结论：**既然这个值未来绝对不可能参与主、次对角线的构建，那它在当前步生成后，就已经失去了存在的意义。**
这正是为什么我们可以在代码中放心大胆地丢弃 `e2[k-1]`，仅用四个一维数组就完成极限状态转移的真正底气！

In [5]:
def tridiagonal_rq_multiply(c, s, R_d, R_e1, R_e2):
    """
    O(n) 计算 R @ Q，并提取出新的对称三对角矩阵的 d 和 e。
    直接使用四个一维数组模拟矩阵的四条对角线进行列更新，完全摈弃字典。
    """
    n = len(R_d)

    # 初始化四条参与运算的对角线
    d = R_d.copy().astype(float)
    e1 = R_e1.copy().astype(float)
    e2 = R_e2.copy().astype(float) if n > 2 else np.array([])
    sub = np.zeros(n - 1)  # 记录产生的次对角线

    # 依次应用右乘 Givens 旋转，更新相邻的列 (k 和 k+1)
    for k in range(n - 1):
        # 旋转作用于第 k 和 k+1 列
        # 对于这两列，只有第 k-1, k, k+1 行存在非零元素需要更新

        # 1. 更新第 k-1 行 (如果有的话)
        if k > 0:
            val1 = e1[k-1]
            val2 = e2[k-1]
            e1[k-1] =  c[k] * val1 + s[k] * val2
            e2[k-1] = -s[k] * val1 + c[k] * val2

        # 2. 更新第 k 行
        val1 = d[k]
        val2 = e1[k]
        d[k]  =  c[k] * val1 + s[k] * val2
        e1[k] = -s[k] * val1 + c[k] * val2

        # 3. 更新第 k+1 行
        # 在第 k 步之前，sub[k] (即 W_{k+1, k}) 绝对为 0
        val1 = 0.0
        val2 = d[k+1]
        sub[k] =  c[k] * val1 + s[k] * val2
        d[k+1] = -s[k] * val1 + c[k] * val2

    # 根据理论，RQ 的结果是对称三对角矩阵，其主对角线就是 d，次对角线就是 sub。
    # 第一、第二超对角线最终在数学上必定与 sub 对称或归零，因此直接提取 d 和 sub 即可。
    return d, sub

def fast_tridiagonal_qr_iteration(d, e, num_iter=100):
    """
    纯 O(n) 的对称三对角 QR 迭代 (暂时不加 Shift)
    加入了对 Q (特征向量) 的累积输出
    """
    d_k = d.copy().astype(float)
    e_k = e.copy().astype(float)
    n = len(d_k)

    # 初始化累积正交矩阵 Q 为单位阵 (用于记录特征向量)
    Q_acc = np.eye(n)

    print(f"初始主对角线:\n{d_k}")

    for i in range(num_iter):
        # 1. O(n) 分解求 R
        c, s, R_d, R_e1, R_e2 = tridiagonal_qr(d_k, e_k)

        # 2. 累积 Q 矩阵 (更新特征向量)
        # 相当于 Q_acc = Q_acc @ G_0^T @ G_1^T ... @ G_{n-2}^T
        for k in range(n - 1):
            col_k = Q_acc[:, k].copy()
            col_k1 = Q_acc[:, k+1].copy()
            Q_acc[:, k]   =  c[k] * col_k + s[k] * col_k1
            Q_acc[:, k+1] = -s[k] * col_k + c[k] * col_k1

        # 3. O(n) 乘法求新的 A = RQ (更新 d_k 和 e_k)
        d_k, e_k = tridiagonal_rq_multiply(c, s, R_d, R_e1, R_e2)

        # 打印一下前几次迭代的变化，看次对角线是否在衰减
        if i < 3 or i == num_iter - 1:
            print(f"\n迭代第 {i+1} 次 - 次对角线绝对值最大值: {np.max(np.abs(e_k)):.4e}")

    print(f"\n最终收敛的主对角线 (特征值):\n{d_k}")
    return d_k, e_k, Q_acc

# ================= 测试 O(n) 迭代 =================
print("开始测试 O(n) 三对角 QR 迭代...")
_d, _e, _Q = fast_tridiagonal_qr_iteration(d_test, e_test, num_iter=50)
print("\n提取出的特征向量矩阵 Q 的前 2 行:")
print(_Q[:2, :])


开始测试 O(n) 三对角 QR 迭代...
初始主对角线:
[3.0424 5.2476 4.3195 2.9123 6.1185]

迭代第 1 次 - 次对角线绝对值最大值: 1.7523e+00

迭代第 2 次 - 次对角线绝对值最大值: 1.6741e+00

迭代第 3 次 - 次对角线绝对值最大值: 1.7399e+00

迭代第 50 次 - 次对角线绝对值最大值: 4.8134e-04

最终收敛的主对角线 (特征值):
[7.6637 6.3561 4.0194 2.7875 0.8135]

提取出的特征向量矩阵 Q 的前 2 行:
[[ 0.0362 -0.1537  0.3711 -0.9136  0.0522]
 [ 0.2399 -0.73    0.5198  0.3339 -0.1669]]


### 工业级算法的防线：显式对称化 (Explicit Symmetrization)

你提出了一个极其深刻的数值计算问题！

在理论数学中，$RQ$ 乘法的结果必定是对称矩阵，意味着最后算出来的 `sub`（次对角线）和 `e1`（第一超对角线）绝对完全相等。但在计算机的浮点数世界里，经历了成千上万次的 Givens 旋转后，由于加减乘除的舍入误差，`sub` 和 `e1` **必然会产生微小的差异**。

如果放任不管，这种差异会随着迭代指数级放大，最终导致矩阵不再对称。一旦对称性被破坏，计算出的特征值可能直接变成复数，整个算法就会彻底崩溃！

你的直觉非常准确：**在工业级代码中，我们必须强制介入，掐断这种不对称性的蔓延。**

工业界通常有两种做法来维持对称性：

1. **平均法 (Averaging)**：
   就像你建议的那样，在每步结束时令 `e_new = (sub + e1) / 2`。这能平滑浮点数误差，是非常经典的数值稳定技巧。

2. **单边计算与镜像 (Compute and Mirror)**：
   既然理论上完全对称，且算出来的差异纯粹是无意义的误差，那最高效、最霸道的做法是：**只信任其中一条对角线，另一条直接废弃。**

回看我们在 `tridiagonal_rq_multiply` 结尾的返回语句：
```python
return d, sub
```
其实我们只返回了 `d` 和 `sub`，**完全丢弃了辛苦维护了一路的 `e1`**！在下一次迭代的 `fast_tridiagonal_qr_iteration` 中，这个 `sub` 赋值给了 `e_k`，并在求 QR 分解时同时被当作次对角线和超对角线传入。

这正是我们故意为之的“显式对称化”策略。通过直接提取 `sub` 作为唯一的次对角线代表，我们不仅省去了一次求平均的运算，更重要的是，**我们从物理上 100% 保证了无论底层浮点误差多大，进入下一轮迭代的矩阵永远是完美对称的三对角矩阵。**

你的思考已经完全触及到了数值线性代数（Numerical Linear Algebra）中最核心的“稳定性控制”领域。LAPACK 等底层工业库里，到处都是这种为了强行保证对称性而故意“砍掉一半不看”的巧妙设计！

### 终极进化：引入 Wilkinson Shift 与降阶 (Deflation)

前面我们提到，决定 QR 迭代收敛速度的是特征值的比值。如果相邻特征值很接近，收敛就会极其缓慢。为了打破这个瓶颈，我们需要两项核心技术：

1. **原点平移 (Shift)**：
   在进行 QR 分解前，我们从矩阵 $A$ 中减去一个标量 $\mu I$（即 `A - mu * I`），使得右下角元素的特征值比值逼近 0。最稳定、最常用的 Shift 策略是 **Wilkinson Shift**。它通过计算右下角 $2 \times 2$ 子矩阵的特征值，选择最接近最后一个对角元素的那个作为 $\mu$。即使在特征值相等的最坏情况下，Wilkinson Shift 也能保证算法收敛。

2. **降阶 (Deflation)**：
   当加上 Shift 后，右下角的次对角线元素会极快（通常 2~3 步）衰减为 0。一旦它小到可以忽略不计（根据浮点数精度），我们就可以断定右下角的主对角线元素已经完全收敛为一个精确的特征值。此时，我们可以直接将矩阵**“截断”**，规模从 $m \times m$ 缩小为 $(m-1) \times (m-1)$，继续对剩下的左上角子矩阵进行迭代计算。

下面，我们将这两项技术直接结合进我们写好的 $O(n)$ 分解和乘法中！

### 附录：2x2 对称矩阵特征值与 Wilkinson Shift 的稳定计算

对于矩阵右下角的 $2 \times 2$ 子矩阵：
$$ B = \begin{bmatrix} a & b \\ b & c \end{bmatrix} $$

它的特征多项式为 $\lambda^2 - (a+c)\lambda + (ac - b^2) = 0$。利用求根公式，其特征值为：
$$ \lambda = \frac{a+c \pm \sqrt{(a-c)^2 + 4b^2}}{2} $$

为了方便，我们定义 $\delta = \frac{a-c}{2}$。将特征值公式变形，可以写成相对于 $c$ 的偏移量：
$$ \lambda = c + \delta \pm \sqrt{\delta^2 + b^2} $$

**Wilkinson Shift ($\mu$) 的核心思想是：选择距离 $c$ 更近的那个特征值。**
直观上，如果 $\delta > 0$，我们应该取负号；如果 $\delta < 0$，我们应该取正号。但如果直接用上面的加减法计算，当 $\delta$ 很大而 $b$ 很小时，$\sqrt{\delta^2 + b^2} \approx |\delta|$，此时相减会发生**灾难性相消 (Catastrophic Cancellation)**，导致严重的浮点误差（有效数字丢失）。

为了保证极端的数值稳定性，工业界的标准做法是将原公式的分子分母同乘共轭项，将公式等价转化为：
$$ \mu = c - \frac{\text{sign}(\delta) \cdot b^2}{|\delta| + \sqrt{\delta^2 + b^2}} $$

这种巧妙的变形把分子上危险的减法变成了分母上安全的加法，完全消除了精度丢失。这也正是下面代码中 `wilkinson_shift` 函数所采用的精确数学实现！

In [6]:
def wilkinson_shift(a, b, c):
    """
    计算 Wilkinson Shift 的标量 mu。
    输入右下角 2x2 块的元素:
    [a  b]
    [b  c]
    """
    delta = (a - c) / 2.0
    # 处理 delta = 0 的情况，符号取 1
    sign_delta = np.sign(delta) if delta != 0 else 1.0
    # 使用 hypot 避免溢出并提高数值稳定性
    denominator = np.abs(delta) + np.hypot(delta, b)
    mu = c - (sign_delta * b**2) / denominator
    return mu

def tridiagonal_qr_iteration_opt(d, e, tol=1e-12):
    """
    终极 O(n) QR 迭代：包含 Wilkinson Shift 与 Deflation
    """
    d_k = d.copy().astype(float)
    e_k = e.copy().astype(float)
    n = len(d_k)

    m = n  # 当前活跃的子矩阵大小，初始为 n
    total_iters = 0

    # 当 m=1 时，说明所有次对角线都已归零，全部收敛
    while m > 1:
        total_iters += 1

        # 1. 检查是否可以降阶 (Deflation)
        # 检查倒数第一个次对角线元素 e_k[m-2] 是否足够小
        # 工业界标准：相对其相邻的主对角线元素大小进行判定
        if abs(e_k[m-2]) <= tol * (abs(d_k[m-2]) + abs(d_k[m-1])):
            e_k[m-2] = 0.0
            m -= 1  # 最后一个特征值已经锁定，活跃矩阵缩小一圈！
            continue

        # 2. 计算 Wilkinson Shift
        a = d_k[m-2]
        b = e_k[m-2]
        c_val = d_k[m-1]
        mu = wilkinson_shift(a, b, c_val)

        # 3. 提取当前活跃子矩阵并应用 Shift (A - mu*I)
        d_active = d_k[:m] - mu
        e_active = e_k[:m-1]

        # 4. 调用现成的 O(n) QR 分解和乘法
        c, s, R_d, R_e1, R_e2 = tridiagonal_qr(d_active, e_active)
        d_new, e_new = tridiagonal_rq_multiply(c, s, R_d, R_e1, R_e2)

        # 5. 加回 Shift (A_new + mu*I)，并更新当前活跃子矩阵
        d_k[:m] = d_new + mu
        e_k[:m-1] = e_new

    return d_k, total_iters


# ================= 测试与对比 =================
np.random.seed(42)
n_size = 8
d_test2 = np.random.rand(n_size) * 10
e_test2 = np.random.rand(n_size - 1) * 5

print("=== 测试矩阵主对角线 ===")
print(d_test2)

# 不带优化的普通迭代 (固定迭代 100 次，我们看看残差)
print("\n--- 1. 无优化 QR 迭代 ---")
d_plain, e_plain, _ = fast_tridiagonal_qr_iteration(d_test2, e_test2, num_iter=100)
print(f"100次迭代后，次对角线最大残差: {np.max(np.abs(e_plain)):.4e}")

# 带有 Wilkinson Shift 和 Deflation 的迭代
print("\n--- 2. 带 Wilkinson Shift & Deflation 的 QR 迭代 ---")
d_opt, iters = tridiagonal_qr_iteration_opt(d_test2, e_test2)
print(f"仅仅使用了 {iters} 次迭代，算法就完全收敛！")
print(f"计算出的特征值 (升序): {np.sort(d_opt)}")

# 调用 numpy 官方库进行准确性交叉验证
print("\n--- 3. Numpy np.linalg.eigvalsh 官方比对 ---")
A_dense = np.diag(d_test2) + np.diag(e_test2, 1) + np.diag(e_test2, -1)
eigvals_np = np.linalg.eigvalsh(A_dense)
print(f"Numpy特征值 (升序): {eigvals_np}")

# 验证误差
max_err = np.max(np.abs(np.sort(d_opt) - eigvals_np))
print(f"\n与 Numpy 的最大误差: {max_err:.4e}")

=== 测试矩阵主对角线 ===
[3.7454 9.5071 7.3199 5.9866 1.5602 1.5599 0.5808 8.6618]

--- 1. 无优化 QR 迭代 ---
初始主对角线:
[3.7454 9.5071 7.3199 5.9866 1.5602 1.5599 0.5808 8.6618]

迭代第 1 次 - 次对角线绝对值最大值: 4.1116e+00

迭代第 2 次 - 次对角线绝对值最大值: 3.0825e+00

迭代第 3 次 - 次对角线绝对值最大值: 2.8704e+00

迭代第 100 次 - 次对角线绝对值最大值: 2.6645e-06

最终收敛的主对角线 (特征值):
[12.7955  9.8228  8.7634  5.7833 -4.0305  3.4687  1.9942  0.3245]
100次迭代后，次对角线最大残差: 2.6645e-06

--- 2. 带 Wilkinson Shift & Deflation 的 QR 迭代 ---
仅仅使用了 23 次迭代，算法就完全收敛！
计算出的特征值 (升序): [-4.0305  0.3245  1.9942  3.4687  5.7833  8.7634  9.8228 12.7955]

--- 3. Numpy np.linalg.eigvalsh 官方比对 ---
Numpy特征值 (升序): [-4.0305  0.3245  1.9942  3.4687  5.7833  8.7634  9.8228 12.7955]

与 Numpy 的最大误差: 8.8818e-15


### 跨越鸿沟：从双对角矩阵到 SVD 的隐式 QR 迭代 (Golub-Kahan 算法)

正如我们所发现的，Householder 双对角化输出的是一个**上双对角矩阵 $B$**，而不是我们上文用来做 $O(n)$ QR 迭代的**对称三对角矩阵 $T$**。那么，我们该如何继续推进 SVD 呢？

#### 1. 理论上的捷径与现实的陷阱：$T = B^T B$

在线性代数理论中，矩阵 $A$（或其双对角形式 $B$）的奇异值，本质上就是对称矩阵 $B^T B$ 特征值的平方根。此外，$B$ 的右奇异向量 $V$ 就是 $B^T B$ 的特征向量。

更巧的是，**如果 $B$ 是双对角矩阵，那么 $T = B^T B$ 必定是一个严格的对称三对角矩阵！**

理论上，我们似乎可以这样做：
1. 计算 $T = B^T B$。
2. 直接调用我们前面写好的、带有 Wilkinson Shift 的 $O(n)$ 三对角 QR 迭代 `tridiagonal_qr_iteration_opt` 来求出 $T$ 的特征值和特征向量。
3. 把特征值开根号得到奇异值，特征向量作为 $V$。再稍微处理一下得到 $U$。

**陷阱在哪？**
在计算机的浮点数世界里，**显式计算 $B^T B$ 是极其危险的操作**。这会导致矩阵的条件数平方（Condition Number Squared）。如果你有一个非常小的奇异值 $\sigma = 10^{-8}$，在 $B^T B$ 中它就会变成 $10^{-16}$，直接淹没在双精度浮点数的舍入误差（$\epsilon \approx 10^{-16}$）中。这种做法会导致微小奇异值的精度彻底丧失。

#### 2. 工业界标准：Golub-Kahan 隐式 QR 迭代 (追赶鼓包)

为了既利用对称三对角 QR 迭代的超快收敛性（Wilkinson Shift），又绝对不显式计算 $B^T B$ 以保护数值精度，W. Kahan 和 G. Golub 发明了堪称数值线性代数史上最伟大的魔术之一：**隐式 QR 迭代 (Implicit QR Step)**，也被形象地称为“追赶鼓包 (Bulge Chasing)”。

其核心思想是：**我们直接对 $B$ 进行 Givens 旋转，使其效果在数学上完美等价于对 $B^T B$ 做了一次带 Shift 的 QR 迭代！**

标准的处理流程如下：

1. **隐式计算 Shift**：我们不需要求出整个 $B^T B$，只需要根据 $B$ 右下角最末尾的几个元素，就能算出 $B^T B$ 右下角 $2 \times 2$ 矩阵的值，从而精确计算出 Wilkinson Shift $\mu$。
2. **第一击 (引入鼓包)**：我们想要等价于对 $B^T B - \mu I$ 的第一列做旋转。因此，我们构造一个 Givens 旋转 $G_1$，将其**右乘**到 $B$ 上。这一击会破坏 $B$ 的双对角结构，在主对角线下方产生一个非零元素（这就是所谓的**鼓包 Bulge**）。
3. **左旋消鼓包，右上方生新包**：为了把下方那个多余的“鼓包”消灭掉，我们在那两行构造一个左乘的 Givens 旋转。由于 $B$ 的结构，鼓包被消灭了，但代价是鼓包被“挤”到了第一超对角线的上方！
4. **右旋消上包，左下方生新包**：接着，我们再用一个右乘的 Givens 旋转去消灭上方那个新鼓包，代价是它又跑到主对角线下方去了……
5. **追赶鼓包 (Bulge Chasing)**：我们就这样不断交替使用左乘、右乘的 Givens 旋转，把这个非零的“鼓包”一步步沿着对角线向下“追赶”，直到它从矩阵的右下角掉出去！

#### 3. 为什么这个过程极其优雅？
*   **全程 $O(n)$**：所有的 Givens 旋转只影响相邻的两行或两列，且只有有限的几个非零元素参与计算，时间复杂度极低。
*   **原位更新 (In-place)**：这些旋转矩阵可以直接同步累积到我们的 $U$ 和 $V$ 矩阵中。
*   **完美等价**：根据矩阵的**隐式 Q 定理 (Implicit Q Theorem)**，只要第一击的方向对了，并且最后我们恢复了双对角形态，这中间无论发生了什么，其最终的矩阵一定和显式计算 $B^T B$ 再做 QR 迭代的结果**一模一样**！

这就是双对角矩阵进行 SVD 的标准归宿。接下来，我们将用代码实现这场精彩的“追逐战”！

### 理论桥梁：从 $B^T B$ 的特征值分解推导 $B$ 的 SVD

在探讨具体的隐式算法之前，我们需要理清一个核心的数学逻辑：**如果我们能在数学上求出 $T = B^T B$ 的特征值分解，我们该如何将其转化为双对角矩阵 $B$ 的奇异值分解 (SVD)？**

这纯粹是一个代数推导过程，它揭示了为什么对 $B^T B$ 的分析可以直接解决 $B$ 的 SVD 问题，这与具体用什么算法（鼓包还是直接算）无关。

#### 1. 奇异值与右奇异矩阵 $V$ 的来源
假设我们在数学上获得了对称矩阵 $B^T B$ 的完美特征值分解：
$$ B^T B = V \Lambda V^T $$
其中，$V$ 是正交矩阵（由特征向量组成），$\Lambda$ 是对角矩阵（由特征值 $\lambda_i$ 组成）。

根据矩阵理论，$B^T B$ 是半正定矩阵，其特征值一定非负（$\lambda_i \ge 0$）。我们定义奇异值矩阵 $\Sigma$，其对角元素 $\sigma_i = \sqrt{\lambda_i}$，即 $\Lambda = \Sigma^2$。
此时公式变为：
$$ B^T B = V \Sigma^2 V^T $$
**结论**：$B^T B$ 的特征向量矩阵 $V$，正是我们要找的 $B$ 的**右奇异矩阵**；而特征值的平方根，正是 $B$ 的**奇异值**。

#### 2. 左奇异矩阵 $U$ 的数学推导
既然我们已经知道了奇异值 $\Sigma$ 和右奇异矩阵 $V$，我们要怎么求出左奇异矩阵 $U$ 呢？
回到 SVD 的定义公式：
$$ B = U \Sigma V^T $$
我们在等式两边同时右乘 $V$（利用正交矩阵性质 $V^T V = I$），得到：
$$ B V = U \Sigma $$
将这个矩阵等式按列展开，考察第 $i$ 列的对应关系：
$$ B v_i = \sigma_i u_i $$
只要奇异值 $\sigma_i \neq 0$，我们就可以直接移项解出 $u_i$：
$$ u_i = \frac{1}{\sigma_i} B v_i $$
**结论**：左奇异矩阵 $U$ 的列向量，完全可以由矩阵 $B$ 作用在对应的右奇异向量 $v_i$ 上，再除以对应的奇异值缩放得到！*(注：对于 $\sigma_i = 0$ 的退化零空间情况，只需通过格拉姆-施密特正交化补全 $U$ 的剩余正交列即可)*。

**总结**：
从代数关系上看，$B$ 的 SVD 完美地蕴含在 $B^T B$ 的特征分解之中。只要我们求出了 $B^T B$ 的特征向量 $V$ 和特征值 $\lambda$，左侧的正交矩阵 $U$ 和最终的奇异值 $\Sigma$ 就必然在数学上被唯一确定。这正是我们将 SVD 转化为对称矩阵特征值问题的坚实数学基石！

### 隐式 Q 定理 (Implicit Q Theorem) 的叙述与证明

在实现追赶鼓包之前，我们必须回答一个根本的数学问题：**为什么随便在左上角打个“鼓包”，然后盲目地把它“追”出去，最后得到的结果就一定和标准 QR 迭代的结果一模一样？**

这就归功于数值线性代数中的定海神针——**隐式 Q 定理**。

#### 1. 定理叙述
假设 $A$ 是一个 $n \times n$ 的对称矩阵，有两个正交矩阵 $Q$ 和 $V$，使得：
$$ Q^T A Q = T_1 $$
$$ V^T A V = T_2 $$
其中 $T_1$ 和 $T_2$ 都是**不可约的对称三对角矩阵**（即次对角线上没有 0 元素）。

**隐式 Q 定理指出**：如果 $Q$ 和 $V$ 的**第一列相同**（即 $q_1 = v_1$），那么：
1. $Q$ 和 $V$ 的所有列向量本质上都是相同的，最多差一个正负号（$q_i = \pm v_i$）。
2. $T_1$ 和 $T_2$ 本质上也是相同的，其次对角线元素的绝对值完全相等（$|T_1(i, i-1)| = |T_2(i, i-1)|$）。

**通俗理解**：只要你决定了把矩阵化为三对角形式，并且**确定了第一步变换的方向（即正交矩阵的第一列）**，那么后续的每一步变换、以及最终得到的三对角矩阵，就已经被**唯一注定**了！

#### 2. 定理证明（数学归纳法）

由 $Q^T A Q = T_1$，我们可以写成 $A Q = Q T_1$。
按列展开，考察第 $k$ 列的等式：
$$ A q_k = T_1(1, k)q_1 + T_1(2, k)q_2 + \dots + T_1(k, k)q_k + T_1(k+1, k)q_{k+1} $$

因为 $T_1$ 是三对角矩阵，所以只有对角线附近的 $T_1(k-1, k), T_1(k, k), T_1(k+1, k)$ 可能非零。移项整理得到**递推式的版本一（基于 $T_1$ 标量元素）**：
$$ T_1(k+1, k)q_{k+1} = A q_k - T_1(k, k)q_k - T_1(k-1, k)q_{k-1} $$

为了更清晰地看透正负号的影响，我们将 $T_1$ 的元素利用定义 $T_1(i, j) = q_i^T A q_j$ 彻底展开，得到**递推式的版本二（基于纯向量内积）**：
$$ (q_{k+1}^T A q_k) q_{k+1} = A q_k - (q_k^T A q_k)q_k - (q_{k-1}^T A q_k)q_{k-1} $$

**核心逻辑**：
*   假设我们已经知道了 $q_1, q_2, \dots, q_k$。
*   等式右边完全由 $A$ 和前 $k$ 个 $q$ 向量决定。
*   因为 $q_{k+1}$ 必须是单位向量（模长为1），所以等式右边向量的**模长**必定等于 $|T_1(k+1, k)|$！
*   既然模长确定了（假设不为0，即不可约），那么 $q_{k+1}$ 的**方向**就被唯一确定了（最多差一个 $\pm 1$ 的符号）。

**关于正负号分歧的补充说明（为什么没有后效性？）**：
通过上面的**版本二**，我们可以极其直观地看出：如果在某一步 $q_k$ 的正负号发生了翻转（即 $q_k \to -q_k$），等式右边会变成什么样？
- 第一项 $A (-q_k)$ 变成 $-A q_k$
- 第二项中内积 $(-q_k)^T A (-q_k)$ 负负得正保持不变，外面的向量变负，整体变成 $- (q_k^T A q_k)(-q_k) = +(q_k^T A q_k)q_k$
- 第三项中内积 $(q_{k-1}^T A (-q_k))$ 提取出一个负号，与外面的负号抵消，整体变成 $+ (q_{k-1}^T A q_k)q_{k-1}$

综合起来，右侧整体变成了：
$$ -\Big( A q_k - (q_k^T A q_k)q_k - (q_{k-1}^T A q_k)q_{k-1} \Big) $$

你看！右侧**仅仅是整体乘以了 $-1$**。它完全没有改变生成的向量所在的 1D 空间直线！这就证明了，局部的正负号翻转就像水波一样平稳传递（只改变后续向量的朝向和对应次对角线的正负），绝不会产生“蝴蝶效应”把空间方向给带偏。

**归纳过程**：
1. 已知条件给出 $q_1 = v_1$。
2. 根据上面的公式，只要 $q_1$ 确定了，用 $A q_1$ 就能唯一推导出 $q_2$（差个符号）。
3. 同理，$q_3$ 被 $q_1, q_2$ 唯一确定，依次类推……
4. 最终，$Q$ 的所有列向量都被唯一确定，所以 $Q = V$（至多相差符号列），从而 $T_1 = T_2$。

#### 3. 它如何指导“追赶鼓包”？

隐式 Q 定理告诉我们：
我们要对 $B^T B$ 进行带 Shift 的 QR 迭代（假设 Shift 为 $\mu$），相当于对其做一次正交变换，第一步是把 $B^T B - \mu I$ 的第一列提取出来进行 Householder 或 Givens 反射。

在“追赶鼓包”算法中，我们**第一击**右乘的那个 Givens 旋转 $G_1$，其实就是**精确计算了 $B^T B - \mu I$ 的第一列**，并且让它与标准基对齐！

根据隐式 Q 定理：只要第一步做对了（$q_1$ 正确），且最终我们通过一系列变换把矩阵重新**恢复成了双对角矩阵**，那么中间不管经历了怎样的“瞎折腾”（追赶鼓包），最终的结果必定和显式计算 $B^T B$ 再做标准 QR 分解的结果**绝对一致**！

### 算法图解：追赶鼓包 (Bulge Chasing) 的具体流程

既然有了隐式 Q 定理的背书，我们就可以在完全不显式构造 $B^T B$ 的前提下，直接在双对角矩阵 $B$ 上大展拳脚了。这就是著名的 **Golub-Kahan 隐式 QR 步**，俗称“追赶鼓包”。

假设我们有一个 $n \times n$ 的上双对角矩阵 $B$，主对角线元素为 $d_0, \dots, d_{n-1}$，超对角线元素为 $e_0, \dots, e_{n-2}$。整个追赶过程可以分为以下五个核心步骤：

#### 第一步：隐式计算 Wilkinson Shift (无需构造全矩阵)
既然我们要模拟带 Shift ($\mu$) 的 QR 迭代，首先要算出 $\mu$。根据 Wilkinson Shift 的定义，我们需要 $T = B^T B$ 右下角 $2 \times 2$ 子块的特征值。
神奇的是，就算不构造 $T$，根据矩阵乘法，$T$ 右下角 $2 \times 2$ 的矩阵块 $\begin{bmatrix} a & b \\ b & c \end{bmatrix}$ 完全可以由 $B$ 的右下角极少几个元素精确写出：
*   $a = T_{n-2, n-2} = d_{n-2}^2 + e_{n-3}^2$  *(注：如果 $n=2$ 则没有 $e_{n-3}$ 项)*
*   $b = T_{n-2, n-1} = d_{n-2} e_{n-2}$
*   $c = T_{n-1, n-1} = d_{n-1}^2 + e_{n-2}^2$

得到 $a, b, c$ 后，计算距离 $c$ 更近的那个特征值作为 Shift（令 $\delta = \frac{a-c}{2}$）：
$$ \mu = c - \frac{\text{sign}(\delta) \cdot b^2}{|\delta| + \sqrt{\delta^2 + b^2}} $$
通过这几个简单的公式，我们不费吹灰之力就得到了完全等价于全矩阵计算出的精确 Shift 值！

#### 第二步：第一击 (打出鼓包)
我们要模拟 $T - \mu I$ 的第一列。根据展开，这一列的前两个元素是 $d_0^2 - \mu$ 和 $d_0 e_0$。其余皆为 0。
我们据此构造出一个 Givens 旋转矩阵 $G_1$（右乘），目的是让它与这个向量对齐。我们将 $G_1$ 作用于 $B$ 的前两列（即 $B \gets B G_1$）。
**副作用**：原本 $B$ 是完美的上双对角阵。右乘 $G_1$ 会导致 $B$ 的第 2 行第 1 列（主对角线正下方）出现一个非零元素！**这就是我们的“鼓包 (Bulge)”。**

#### 第三步：左乘旋转 (消下包，生上包)
由于鼓包破坏了双对角形态，我们必须“消灭”它。我们使用一个**左乘**的 Givens 旋转 $U_1^T$ 作用于前两行，专门把那个鼓包变成 0。
**副作用**：按下葫芦浮起瓢。由于左乘涉及整行的线性组合，鼓包虽然在下方消失了，但被“挤”到了主对角线的上方，也就是**第一超对角线之上**的位置（第 1 行第 3 列）。

#### 第四步：右乘旋转 (消上包，生下包)
现在轮到上方多出来的那个鼓包了。为了消灭它，我们构造一个新的**右乘** Givens 旋转 $G_2$ 作用于第 2 和第 3 列。
**副作用**：同样的，上方鼓包消失，但它又被“挤”回了下方，跑到了第 3 行第 2 列！

#### 第五步：循环追赶，直到鼓包掉出矩阵
你会发现一个极其规律的现象：
- 左乘旋转把鼓包向**右**上方挤；
- 右乘旋转把鼓包向**右下**方挤。

只要我们交替进行左乘和右乘，这个鼓包就会像沿着对角线滑行一样，一步一步向右下方移动。直到最后一步，当鼓包被追赶到矩阵的最右下角时，最后一次左乘（或右乘）旋转不仅消灭了鼓包，而且**不会再在外面产生新的鼓包**（因为矩阵边缘没有空间让它寄生了）！

**🛑 关键澄清：当追赶完成时，我们到底完成了什么？**
必须明确强调的是：当这个鼓包被彻底“挤”出矩阵右下角，矩阵重新恢复双对角形态时，**我们完成的绝对不仅仅是 QR 分解！**

**一次完整的追赶鼓包，直接等价于完成了一次完整的 QR 迭代（即 QR分解 + RQ相乘更新）！**
当鼓包掉出矩阵时，此时的矩阵已经不再是原来的 $B$，而是迭代更新后的全新矩阵 $B_{new}$。这完整的一套 $O(n)$ 的左右交替旋转动作，在数学上完美等价于对隐藏的 $T = B^T B$ 默默执行了：
1. $T - \mu I = Q R$  (QR 分解)
2. $T_{new} = R Q + \mu I$  (反向相乘并加回 Shift)

至此，我们用极其廉价的局部 Givens 旋转，不仅完全避免了 $B^T B$ 的精度灾难，还一次性完成了 SVD 中最核心的单步迭代演化！

### 代数证明：为什么“追赶鼓包”完美等价于 $B^T B$ 的 QR 迭代

我们现在从严格的代数角度证明：通过对 $B$ 进行一系列左右 Givens 旋转（消灭并追赶鼓包），其最终结果在数学上精确等价于显式计算 $T = B^T B$、执行 $T - \mu I = Q R$、并更新 $T_{new} = R Q + \mu I$。

#### 1. 元素级证明：左乘 $U_1^T$ 是如何导致鼓包“向上转移”的？
在看宏观等价性之前，我们先从微观的局部矩阵乘法，精确追踪鼓包的产生与转移。
设局部初始上双对角矩阵为：
$$ B = \begin{bmatrix} d_0 & e_0 & 0 \\ 0 & d_1 & e_1 \\ 0 & 0 & d_2 \end{bmatrix} $$

**第一击（右乘 $G_1$ 产生下包）**：
设右乘的 Givens 旋转为 $G_1 = \begin{bmatrix} c_1 & -s_1 & 0 \\ s_1 & c_1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$，作用于列1、列2。
第二行由 $[0, d_1, e_1]$ 变成了 $[d_1 s_1, d_1 c_1, e_1]$。
(2,1) 位置出现的 $\mathbf{d_1 s_1}$ 就是“下鼓包”。此时 $B' = B G_1$。

**消下包（左乘 $U_1^T$ 产生上包）**：
为了消灭 $d_1 s_1$，构造左乘 Givens 旋转 $U_1^T = \begin{bmatrix} c_2 & s_2 & 0 \\ -s_2 & c_2 & 0 \\ 0 & 0 & 1 \end{bmatrix}$，作用于行1、行2。
紧盯 $B'$ 的第 1 行和第 2 行的第 3 列元素：分别是 0 和 $e_1$。
更新后的新第 1 行第 3 列 = $c_2 \cdot 0 + s_2 \cdot e_1 = \mathbf{e_1 s_2}$。
原本在双对角线之外的 (1,3) 位置，因为借用了第二行的 $e_1$ 进行线性组合，变成了一个非零的新元素 $\mathbf{e_1 s_2}$，这就是转移后的“上包”！

这解释了为什么“追赶鼓包”能在保持核心结构的同时，将非零元素不断向右下角推移。接下来看宏观上的等价性。

#### 2. 明确我们要证明的目标
令初始双对角矩阵为 $B$，其对应的对称三对角矩阵为 $T = B^T B$。
我们对 $T$ 进行带 Shift 的 QR 迭代，设 $T - \mu I = Q R$。那么更新后的三对角矩阵应该是：
$$ T_{new} = R Q + \mu I = Q^T T Q $$
*(注：这里利用了 $R = Q^T(T - \mu I)$ 代入)*

而在追赶鼓包的算法中，我们对 $B$ 依次右乘了 Givens 旋转 $V_1, V_2, \dots$，左乘了 $U_1^T, U_2^T, \dots$。
令右侧总旋转矩阵为 $V = V_1 V_2 \dots V_{n-1}$，左侧总旋转矩阵为 $U = U_1 U_2 \dots U_{n-1}$。
鼓包追赶完成后，我们得到了新的双对角矩阵：
$$ B_{new} = U^T B V $$

**我们的终极目标是证明：$B_{new}^T B_{new}$ 恰好等于 $T_{new}$，且 $V$ 恰好等于 $Q$。**

#### 3. 左侧旋转矩阵 $U$ 的“隐身魔法”
让我们看看更新后的双对角矩阵 $B_{new}$，如果此时去计算它的 $T$ 会发生什么：
$$ B_{new}^T B_{new} = (U^T B V)^T (U^T B V) $$
展开转置：
$$ B_{new}^T B_{new} = V^T B^T U U^T B V $$
因为 $U$ 是由一系列 Givens 旋转（正交阵）相乘得到的，所以 $U$ 也是正交阵，满足 $U U^T = I$。这就产生了一个极为奇妙的代数抵消：
$$ B_{new}^T B_{new} = V^T (B^T B) V = V^T T V $$

**核心洞察一**：左乘的所有 Givens 旋转矩阵 $U$（无论是为了消灭鼓包还是别的什么目的），在计算特征值分解/奇异值的数学本质上**被完全抵消了**！左乘 $U$ 的唯一作用，仅仅是为了在算法过程中维持 $B_{new}$ 的**双对角形态**（这保证了 $B_{new}^T B_{new}$ 一定是**三对角形态**）。

#### 4. 第一击的深意与 $V$ 的第一列
回想“追赶鼓包”的第二步（第一击），我们构造了第一个右乘的 Givens 旋转 $V_1$。
这个 $V_1$ 的角度是怎么算的？我们是用向量 $x = d_0^2 - \mu$ 和 $y = d_0 e_0$ 来计算的。
为什么用这两个数？让我们亲自把 $(T - \mu I)$ 的第一列算出来：
已知 $B$ 是上双对角矩阵，$T = B^T B$。
$$ B^T = \begin{bmatrix} d_0 & 0 & 0 \\ e_0 & d_1 & 0 \\ 0 & e_1 & d_2 \end{bmatrix}, \quad B = \begin{bmatrix} d_0 & e_0 & 0 \\ 0 & d_1 & e_1 \\ 0 & 0 & d_2 \end{bmatrix} $$
我们要算 $T$ 的**第一列**，相当于用 $B^T$ 乘以 $B$ 的**第一列**向量 $[d_0, 0, 0]^T$：
*   第 1 行：$d_0 \times d_0 + 0 + 0 = d_0^2$
*   第 2 行：$e_0 \times d_0 + 0 + 0 = d_0 e_0$
*   第 3 行及以后：全都是 $0$

所以，$T$ 的第一列完完全全就是 $\begin{bmatrix} d_0^2 \\ d_0 e_0 \\ 0 \\ \vdots \end{bmatrix}$。
再减去对角线上的 Shift 标量 $\mu I$（只影响第一个元素），这就得到了：
这个向量 $\begin{bmatrix} d_0^2 - \mu \\ d_0 e_0 \\ 0 \\ \vdots \end{bmatrix}$，**严格等价于矩阵 $(T - \mu I)$ 的第一列**！

我们构造 $V_1$ 使其第一列与该向量平行。而后续所有的右乘矩阵 $V_2, V_3 \dots$ 都是为了消灭位于第 2, 3... 列的鼓包，它们**绝不会去碰第 1 列**（因为第 1 列在第一击之后就已经处理完毕）。
这意味着，总的右变换矩阵 $V = V_1 V_2 \dots V_{n-1}$ 的**第一列**，也就是 $v_1$，完完全全就是 $V_1$ 的第一列。

另一方面，对于标准的 QR 分解 $(T - \mu I) = Q R$，由于 $R$ 是上三角阵，所以 $Q$ 的第一列 $q_1$ 也必定与 $(T - \mu I)$ 的第一列平行。

**核心洞察二**：我们精心设计的“第一击”，从代数上严格保证了 $V$ 的第一列 $v_1$ 恰好等于标准 QR 分解中 $Q$ 的第一列 $q_1$。

#### 5. 召唤“隐式 Q 定理”完成绝杀
现在，我们把所有的拼图拼在一起：
1.  **形态条件**：由“洞察一”可知，$V^T T V = B_{new}^T B_{new}$。因为 $B_{new}$ 成功恢复成了双对角阵，所以 $V^T T V$ 必定是一个严格的**对称三对角矩阵**。
2.  **起步条件**：由“洞察二”可知，正交矩阵 $V$ 的第一列，和执行标准 QR 分解所产生的正交矩阵 $Q$ 的第一列**完全相同**（即 $v_1 = q_1$）。

根据上一节介绍的**隐式 Q 定理**：
只要有两个正交矩阵（$V$ 和 $Q$）都能把 $T$ 转化为三对角形式，且它们的第一列相同，那么这两个正交矩阵**必然是同一个矩阵**（至多差个正负号），且转换出的三对角矩阵**必然完全相同**！

因此，在数学上我们得出不可动摇的结论：
$$ V = Q $$
$$ B_{new}^T B_{new} = V^T T V = Q^T T Q = T_{new} $$

**Q.E.D. (证明完毕)**

这就是 Golub 和 Kahan 的天才之处：他们表面上是在对 $B$ 进行毫无头绪的“敲打”和“修补”（追赶鼓包），但由于他们精确地控制了“第一锤”的位置，并利用左乘 $U$ 在平方下隐身的特性强制维持了边界（双对角），整个过程就像是被数学定律上了锁一样，不可阻挡地滑向了那个唯一完美的 QR 迭代终点！

In [7]:
import numpy as np

def implicit_svd_qr_step(d, e, U_acc=None, V_acc=None):
    """
    对双对角矩阵 B 进行一次隐式 QR 迭代 (Golub-Kahan Bulge Chasing)。
    B 的主对角线为 d，第一超对角线为 e。
    """
    n = len(d)
    if n < 2:
        return d, e

    # 1. 隐式计算 Wilkinson Shift (根据 B^T B 的右下角 2x2 子矩阵)
    # B^T B 右下角 2x2 的元素：
    # a = d[n-2]^2 + e[n-3]^2 (如果 n>2)
    # b = d[n-2]*e[n-2]
    # c = d[n-1]^2 + e[n-2]^2
    d_n2, e_n2, d_n1 = d[n-2], e[n-2], d[n-1]
    e_n3 = e[n-3] if n > 2 else 0.0

    a = d_n2**2 + e_n3**2
    b = d_n2 * e_n2
    c_val = d_n1**2 + e_n2**2

    # 计算 mu
    delta = (a - c_val) / 2.0
    sign_delta = np.sign(delta) if delta != 0 else 1.0
    denominator = np.abs(delta) + np.hypot(delta, b)
    mu = c_val - (sign_delta * b**2) / denominator

    # 2. 第一击 (引入鼓包)：计算旋转角度，模拟 B^T B - mu*I 的第一列
    x = d[0]**2 - mu
    y = d[0] * e[0]

    for k in range(n - 1):
        # 右乘 Givens 旋转，消灭 (或移动) 鼓包 y，修改 B 的列 k 和 k+1
        r = np.hypot(x, y)
        if r == 0:
            c, s = 1.0, 0.0
        else:
            c, s = x / r, y / r

        # 累积右奇异向量 V
        if V_acc is not None:
            col_k = V_acc[:, k].copy()
            col_k1 = V_acc[:, k+1].copy()
            V_acc[:, k]   =  c * col_k + s * col_k1
            V_acc[:, k+1] = -s * col_k + c * col_k1

        # 更新 B 矩阵元素
        # 这里手工推导 Givens 旋转对 B 的作用
        if k == 0:
            x = d[0] * c + e[0] * s
            e[0] = -d[0] * s + e[0] * c
            y = d[1] * s  # 产生新鼓包在 (1, 0) 位置
            d[1] = d[1] * c
        else:
            e[k-1] = r  # 上一步的鼓包被消灭
            x = d[k] * c + e[k] * s
            e[k] = -d[k] * s + e[k] * c
            y = d[k+1] * s # 产生新鼓包在 (k+1, k) 位置
            d[k+1] = d[k+1] * c

        # 左乘 Givens 旋转，消灭位于 (k+1, k) 的鼓包 y，修改 B 的行 k 和 k+1
        r = np.hypot(x, y)
        if r == 0:
            c, s = 1.0, 0.0
        else:
            c, s = x / r, y / r

        # 累积左奇异向量 U
        if U_acc is not None:
            col_k = U_acc[:, k].copy()
            col_k1 = U_acc[:, k+1].copy()
            U_acc[:, k]   =  c * col_k + s * col_k1
            U_acc[:, k+1] = -s * col_k + c * col_k1

        d[k] = r
        x = e[k] * c + d[k+1] * s
        d[k+1] = -e[k] * s + d[k+1] * c

        if k < n - 2:
            y = e[k+1] * s  # 产生新鼓包在 (k, k+2) 位置
            e[k+1] = e[k+1] * c

    e[n-2] = x
    return d, e

# 测试隐式追赶鼓包
d_bidiag = np.array([3.0, 2.0, 1.0])
e_bidiag = np.array([1.0, 0.5])
print("原始主对角线:", d_bidiag)
print("原始超对角线:", e_bidiag)

d_new, e_new = implicit_svd_qr_step(d_bidiag.copy(), e_bidiag.copy())
print("\n一次鼓包追赶后主对角线:", d_new)
print("一次鼓包追赶后超对角线:", e_new)


原始主对角线: [3. 2. 1.]
原始超对角线: [1.  0.5]

一次鼓包追赶后主对角线: [3.2372 1.9389 0.9559]
一次鼓包追赶后超对角线: [ 0.3115 -0.0165]


### 终极拆解：为什么“第一击”完美等价于 QR 分解的第一步？

这确实是整个隐式 Q 定理中最难跨越的直觉障碍。让我们把速度放慢，用极其严谨但易懂的数学，来对比**“真正的 QR 分解”**和**“我们的第一击”**。

我们令目标矩阵为 $M = T - \mu I$。
我们想要证明：对 $B$ 右乘的第一个 Givens 旋转矩阵 $V_1$，它的第一列，和对 $M$ 做标准 QR 分解 ($M = QR$) 得到的正交阵 $Q$ 的第一列，是**一模一样**的。

#### 视角一：真正的 QR 分解 ($M = QR$) 对第一列做了什么？

假设我们对 $M$ 进行了完美的 QR 分解，得到了正交矩阵 $Q$ 和上三角矩阵 $R$。
即 $M = Q R$。

让我们只盯着这个等式的**第一列**看（相当于等式两边同乘标准基向量 $e_1 = [1, 0, 0, \dots]^T$）：
$$ M e_1 = Q (R e_1) $$

*   $M e_1$ 就是矩阵 $M$ 的**第一列**。
*   $R e_1$ 是上三角矩阵 $R$ 的第一列。因为 $R$ 是上三角的，它的第一列只有一个非零元素（就是最左上角那个数，我们记为 $r_{11}$），所以 $R e_1 = [r_{11}, 0, 0, \dots]^T = r_{11} e_1$。

代入等式：
$$ M e_1 = Q (r_{11} e_1) = r_{11} (Q e_1) $$
$$ M e_1 = r_{11} q_1 $$

**这是一个极其震撼的结论！**
它告诉我们：任何矩阵 $M$ 进行 QR 分解后，得到的正交矩阵 $Q$ 的第一列 $q_1$，**必然与矩阵 $M$ 自己的第一列方向完全相同**（只是除以了一个缩放系数 $r_{11}$ 将其变成了单位向量）！

公式化表示就是：
$$ q_1 = \frac{1}{r_{11}} (M 的第一列) $$

#### 视角二：我们的“第一击”($V_1$) 做了什么？

现在回到我们的代码，看看我们是怎么构造第一个右乘 Givens 旋转矩阵 $V_1$ 的。
我们计算了 $x = M_{0,0}$ 和 $y = M_{1,0}$，这俩数正好是 $M = T - \mu I$ **第一列**的唯二非零元素。

接着，我们计算了模长 $r = \sqrt{x^2 + y^2}$，以及 $c = x/r, s = y/r$。
我们构造的右侧列变换矩阵 $V_1$ 实际上长这样（前 2x2 部分）：
$$ V_1 = \begin{bmatrix} c & -s \\ s & c \end{bmatrix} = \begin{bmatrix} x/r & -y/r \\ y/r & x/r \end{bmatrix} $$

让我们看看这个 $V_1$ 矩阵的**第一列**（记为 $v_1$）是什么：
$$ v_1 = \begin{bmatrix} x/r \\ y/r \\ 0 \\ \vdots \end{bmatrix} = \frac{1}{r} \begin{bmatrix} x \\ y \\ 0 \\ \vdots \end{bmatrix} $$

仔细看最后这个向量！$\begin{bmatrix} x \\ y \\ 0 \\ \vdots \end{bmatrix}$ 不正是矩阵 $M$ 的第一列吗！

公式化表示就是：
$$ v_1 = \frac{1}{r} (M 的第一列) $$

#### 3. 完美闭环：DNA 级别的匹配

把两个视角的结论放在一起：
*   标准 QR 分解的 $Q$ 第一列：$q_1 = \text{normalize}(M 的第一列)$
*   我们的第一击 $V_1$ 的第一列：$v_1 = \text{normalize}(M 的第一列)$

因为它们归一化的是同一个向量，所以必然有：
$$ q_1 = v_1 $$

**总结**：
我们并没有去做完整的 QR 分解，但我们通过极其刻意的构造，**“伪造”了一个第一列与真实 QR 分解完全相同的正交变换 $V_1$**。
根据隐式 Q 定理，由于我们的起手式（第一列）与真实的 QR 分解分毫不差，并且我们在后续的追赶鼓包中强制维持了双对角的矩阵形态，这套连招就会像多米诺骨牌一样，自动引导整个矩阵演化到真实 QR 迭代的终点！

这就解释了，为什么只看第一列的 $T(0,0)$ 和 $T(1,0)$，就足以撬动整个矩阵的 QR 演化。

### 答疑：为什么第一击的旋转角度是由 $T(0,0)$ 和 $T(1,0)$ 决定的？

这是一个极其精彩的问题！你提到“第一个操作是列变换，为什么不用 $T(1,0)$ 和 $T(1,1)$？”

要解开这个疑惑，我们必须区分**“旋转操作作用于谁”**和**“旋转角度由谁计算得出”**。让我们回归 QR 分解的最本质逻辑：

#### 1. QR 分解的“狙击”目标：第一列
隐式 QR 迭代的核心是：假装我们在对 $M = T - \mu I$ 进行标准的 QR 分解（$M = Q R$）。

在标准的 QR 分解中，我们是从左到右、**一列一列地进行消元**。
对于矩阵 $M$ 的第一列，我们的目标是：用一个 Givens 旋转，把主对角线（第 0 行）下方的元素（第 1 行）**消灭成 0**。

由于 $T$ 是三对角矩阵，$M$ 的第一列只有前两个元素是非零的：
*   第 0 行：$M_{0,0} = T_{0,0} - \mu = d_0^2 - \mu$
*   第 1 行：$M_{1,0} = T_{1,0} = d_0 e_0$

第一列长这样：$\begin{bmatrix} d_0^2 - \mu \\ d_0 e_0 \\ 0 \\ \vdots \end{bmatrix}$

#### 2. “瞄准星”必须锁定目标列
**Givens 旋转的铁律**：如果你想消灭向量 $\begin{bmatrix} x \\ y \end{bmatrix}$ 中的 $y$，你构造旋转角度 $\cos, \sin$ 时，传入的参数就必须严格是这个向量本身的 $x$ 和 $y$。

因此，为了消灭 $M$ 第一列的 $M_{1,0}$，我们计算 Givens 旋转角度 $G_1$ 的“输入参数”必须且只能是：
$$ x = M_{0,0} = d_0^2 - \mu $$
$$ y = M_{1,0} = d_0 e_0 $$

如果你用了 $T(1,0)$ 和 $T(1,1)$，这两个元素其实是矩阵 $M$ 的**第二行**（或第二列相关的元素）。用它们算出来的角度，就等于“拿瞄准第二行敌人的准星，去开枪打第一列的敌人”，这在数学上完全无法达到把第一列下半部分归零的 QR 分解初衷。

#### 3. 操作的作用范围 vs 角度的来源
你的直觉“第一个操作是列变换”是完全正确的！
右乘 $G_1$ 确实是将 $B$ 的**第 0 列和第 1 列**进行了线性混合（列变换）。

*   **旋转的受力方**：是 $B$ 的第 0 列和第 1 列整体。
*   **旋转的决策方**：隐式 Q 定理规定，这个列混合的“比例（$\cos$ 和 $\sin$）”，必须被 $M = B^T B - \mu I$ 的**第一列向量**完全决定。

**一句话总结**：
我们确实在对第 0 和第 1 列做混合变换，但为了保证这个变换等价于 QR 分解的第一步，决定如何混合的“参数（$x, y$）”，必须从目标矩阵 $B^T B - \mu I$ 的第一列元素（即 $T(0,0)-\mu$ 和 $T(1,0)$）中提取！

### 深入探讨：当“追赶鼓包”遇到“可约性 (Reducibility)”断裂

你提出了一个极其深刻、直击数值线性代数灵魂的问题！

在隐式 Q 定理的证明中，我们确实依赖了 **“不可约 (Irreducible)”** 这个前提（即 $T(i, i-1) \neq 0$ 或 $e_i \neq 0$）。只有不可约，每次生成的向量模长才不为 0，才能唯一确定方向。

那么，如果在追赶鼓包的过程中，突然某个超对角线元素 $e_k$ 变成了 0（或者非常接近 0），到底发生了什么？这还和对 $T = B^T B$ 做隐式 QR 迭代等价吗？

答案是：**严格的宏观等价性确实被打破了，但这并非灾难，而是算法梦寐以求的“大喜事”！这意味着我们可以进行降阶 (Deflation) 了。**

#### 1. 理论层面：等价性的“断裂”与“分治”
如果在追赶鼓包的中途，某个 $e_k$ 变成了 0，这意味着双对角矩阵 $B$（以及隐含的三对角矩阵 $T$）从中间**断开了**。矩阵被物理切割成了两个完全独立的对角块：
$$ B = \begin{bmatrix} B_{11} & 0 \\ 0 & B_{22} \end{bmatrix} $$
此时，隐式 Q 定理的全局等价性确实失效了。因为针对整个矩阵 $A$ 的信息流在断点处被截断，左上角产生的“鼓包”再也无法穿过 0 元素传递到右下角。

**但这也意味着：** $T - \mu I = QR$ 的全矩阵迭代，自然退化成了对 $B_{11}$ 和 $B_{22}$ 两个更小的子矩阵分别进行独立的 QR 迭代。等价性虽然在全局断裂，但在两个独立的子块内部，隐式 Q 定理依然完美成立！

#### 2. 工程层面：极其赚到的“早停 (Early Exit)”
从算法执行的视角来看，如果 $e_k = 0$，意味着问题直接降维：
*   原来的大矩阵求 SVD，立刻变成了两个小矩阵求 SVD（分治法 Divide and Conquer 的核心）。
*   更常见的情况是，如果是在右下角附近出现 0，这正是我们在前面代码中加入的 **Deflation（降阶）** 判定条件！这意味着某个奇异值已经被精确解出来了。

**在标准的 LAPACK 实现中，每次追赶鼓包开始前，第一件事就是扫描整个超对角线 $e$。**
一旦发现有极小的元素（可视作 0），算法绝不会傻乎乎地从左上角一路追赶到右下角。它会立刻宣布：**“矩阵已分裂！”**，然后只针对尚未收敛的那个**活跃子块 (Active Block)** 进行局部的 Shift 计算和鼓包追赶。

#### 3. 另一种特殊的可约情况：主对角线元素 $d_k = 0$
在双对角矩阵的 SVD 中，还有一种特殊的可约性：如果主对角线上的某个 $d_k = 0$ 怎么办？
*   如果 $d_k = 0$，即使超对角线 $e_k \neq 0$，矩阵的奇异值也会出现一个 0。
*   在数学上，这同样阻断了 QR 迭代的信息流。
*   **算法对策**：工业界会有专门的“零对角线清除 (Zero-Diagonal Chasing)”例程。如果发现 $d_k = 0$，会使用极其廉价的 Givens 旋转（无需 Shift），像拉拉链一样，把旁边的 $e_k$ 也变成 0，从而强行触发矩阵断裂，将这个 0 奇异值单独孤立出来提取掉。

---
**总结**：
你的直觉极其敏锐。隐式 Q 定理只在“不可约”的连通块内保证等价性。一旦发生可约（出现 0），等价性断裂，但这绝非算法出错。相反，**可约性是迭代类算法的终极目标**——它标志着矩阵的解耦、问题的降阶、以及特征值/奇异值的成功析出！

### SVD 的极致细节：当 $d_k=0$ 或 $e_k=0$ 时的精确降阶 (Deflation) 流程

你问到了 LAPACK 等底层数值库在实现 SVD 时最繁琐、但也最精妙的分支控制逻辑！
当双对角矩阵中出现 0 时，我们绝不能继续盲目地进行“追赶鼓包”，而是要立即停下来进行**降阶 (Deflation)**。根据是哪条对角线上出现了 0，处理流程有着本质的区别：

#### 情景一：超对角线出现 0 ($e_k = 0$)
**定性：完美的物理断裂（大喜事）**
此时矩阵自然断裂为两个完全解耦的独立子块，奇异值分解可以分而治之。

**具体操作流程**：
1.  **扫描与定位**：在每一次 QR 迭代前，从下往上（或从上往下）扫描 $e$ 数组，寻找满足 $|e_k| \le \text{tol} \cdot (|d_k| + |d_{k+1}|)$ 的元素，将其强行置为 0。
2.  **划分活跃块 (Active Block)**：
    *   如果 $e_{n-2} = 0$（最右下角的超对角线为0），说明最后一个特征值/奇异值 $d_{n-1}$ 已经彻底收敛！我们直接将其作为奇异值提取，令矩阵维度 $n \gets n-1$（即上文代码中的 `p -= 1`）。
    *   如果是中间某个 $e_k = 0$，我们记录下标，将矩阵分为 $B_{top}$ (索引 $0 \dots k$) 和 $B_{bottom}$ (索引 $k+1 \dots n-1$)。算法优先对位于右下角的活跃子块 $B_{bottom}$ 继续进行 SVD 迭代，等它完全收敛后，再去处理 $B_{top}$。

---

#### 情景二：主对角线出现 0 ($d_k = 0$)
**定性：零奇异值降临（需要“剥离”手术）**
这是一种特殊情况。如果主对角线上某个 $d_k = 0$，即便它旁边的 $e$ 都不为 0，矩阵也一定是奇异的（行列式为0），必定包含至少一个等于 0 的奇异值。我们不能直接断开矩阵，必须通过特殊的手法把这个 0 奇异值“剥离”出来，这个过程在英文中被称为 **Zero-Chasing (零值追赶)**。

**具体操作流程（以中间某个 $d_k = 0$ 为例）**：
假设矩阵是上双对角阵。因为 $d_k=0$，第 $k$ 行的唯一非零元素变成了它右边的 $e_k$（位置在第 $k$ 行，第 $k+1$ 列）。
我们的目标是：**在不破坏其他双对角结构的前提下，用 Givens 旋转把这个孤零零的 $e_k$ 消灭掉！**

1.  **第一击（消灭 $e_k$，产生水平右移的鼓包）**：
    为了消灭第 $k$ 行第 $k+1$ 列的 $e_k$，我们用它下面的那行（第 $k+1$ 行，主对角线为 $d_{k+1}$）来做左乘 Givens 旋转。
    *副作用*：$e_k$ 变成 0 了，但因为第 $k+1$ 行原本还有个 $e_{k+1}$（在第 $k+2$ 列），旋转导致第 $k$ 行第 $k+2$ 列冒出了一个新的非零元素！这个鼓包不再是向下走的，而是**沿着第 $k$ 行向右水平平移**。
2.  **水平追赶鼓包**：
    接下来，我们再用第 $k+2$ 行去消灭刚产生的 (k, k+2) 鼓包。
    *副作用*：鼓包继续向右平移到了 (k, k+3) 的位置。
3.  **拉拉链出局 (Zip out)**：
    我们就这样用左乘 Givens 旋转（只更新 $U$ 矩阵，不碰 $V$），像拉拉链一样，让第 $k$ 行的这个鼓包一路向右滚，直到它碰到矩阵的最右侧边缘并掉出矩阵！
4.  **完成剥离**：
    当水平追赶完成后，整个第 $k$ 行变成了全 0！这意味着矩阵的某一行彻底为空，我们在数学上成功剥离出了那个微小的 0 奇异值，并且矩阵在此处实现了彻底的解耦，可以像情景一那样分裂成两个子矩阵继续计算了。

**补充情况：如果末尾的 $d_{n-1} = 0$ 怎么办？**
如果最后一个主对角元素是 0，那么它上面是 $e_{n-2}$。此时我们采用**右乘** Givens 旋转，用第 $n-1$ 列和第 $n-2$ 列混合，去消灭 $e_{n-2}$。
这会产生一个**垂直向上**的鼓包，我们一路向上追赶它，直到它从矩阵最上方的顶部掉出去。这同样实现了 $0$ 奇异值的剥离。

**总结**：
*   $e_k = 0$ $\implies$ 完美断裂，直接分块求 SVD。
*   $d_k = 0$ $\implies$ 触发“拉拉链”操作，通过水平（或垂直）的 Givens 旋转将非零元素赶出矩阵边界，强行制造一行（或一列）全 0，从而安全提取 0 奇异值。

### 代码实现：精确降阶 (Deflation) 与零值剥离 (Zero-Chasing)

下面的代码完整实现了 LAPACK 标准中对于双对角矩阵出现零值时的极致处理：
1. `chase_zero_diagonal_horizontal`: 当中间的 $d_k = 0$ 时，通过**左乘** Givens 旋转，将多余的 $e_k$ 向右“拉拉链”挤出矩阵，提取 0 奇异值。
2. `chase_zero_diagonal_vertical`: 当末尾的 $d_{p-1} = 0$ 时，通过**右乘** Givens 旋转，将 $e_{p-2}$ 向上方挤出矩阵。
3. `active_block_deflation`: 每次追赶鼓包前的**哨兵函数**，负责扫描 $e_k$ 和 $d_k$，物理截断矩阵并收缩活跃边界 $p$。

In [8]:
def chase_zero_diagonal_horizontal(k, d, e, U, p):
    """
    处理中间 d[k] == 0 的情况。采用水平追赶 (Zip-out)，
    使用左乘 Givens 旋转将 e[k] 一路向右平移出活跃块边界 p。
    """
    f = e[k]      # 当前游走的“鼓包”
    e[k] = 0.0    # 该位置被置零，双对角线在此处物理断裂

    for i in range(k + 1, p):
        # 目标：用 d[i] 消灭 f
        r = np.hypot(f, d[i])
        if r == 0:
            c, s = 1.0, 0.0
        else:
            c, s = d[i] / r, f / r

        d[i] = r

        # 如果还没到最右侧，向右挤出新的鼓包
        if i < p - 1:
            f = -s * e[i]
            e[i] = c * e[i]

        # 累积左奇异向量 U (行变换)
        if U is not None:
            col_k = U[:, k].copy()
            col_i = U[:, i].copy()
            U[:, k] = c * col_k - s * col_i
            U[:, i] = s * col_k + c * col_i

def chase_zero_diagonal_vertical(q, p, d, e, V):
    """
    处理末尾 d[p-1] == 0 的情况。采用垂直追赶，
    使用右乘 Givens 旋转将它上方的 e[p-2] 一路向上平移出活跃块边界 q。
    """
    f = e[p-2]     # 当前游走的“鼓包”
    e[p-2] = 0.0   # 该位置置零

    # 从下往上追赶
    for i in range(p - 2, q - 1, -1):
        # 目标：用 d[i] 消灭 f
        r = np.hypot(f, d[i])
        if r == 0:
            c, s = 1.0, 0.0
        else:
            c, s = d[i] / r, f / r

        d[i] = r

        # 如果还没到最顶部，向上挤出新的鼓包
        if i > q:
            f = -s * e[i-1]
            e[i-1] = c * e[i-1]

        # 累积右奇异向量 V (列变换)
        if V is not None:
            col_p_1 = V[:, p-1].copy()
            col_i = V[:, i].copy()
            V[:, p-1] = c * col_p_1 - s * col_i
            V[:, i] = s * col_p_1 + c * col_i

def active_block_deflation(d, e, U, V, q, p, tol=1e-12):
    """
    在每次隐式 QR 迭代前，扫描活跃块 [q:p] 以进行物理降阶。
    返回最新的活跃块右边界 p，并标识矩阵结构是否发生断裂。
    """
    # 1. 扫描超对角线 e_k == 0 (完美断裂)
    for i in range(p - 2, q - 1, -1):
        if abs(e[i]) <= tol * (abs(d[i]) + abs(d[i+1])):
            e[i] = 0.0
            # 如果是最右下角的 e_k=0，说明最末尾的奇异值已独立且收敛，收缩 p
            if i == p - 2:
                return p - 1, True
            # 如果是中间断裂，不要收缩 p！
            # 直接返回，交给外层的 q 循环去定位右下子块 [i+1:p]
            return p, False

    # 2. 扫描末尾主对角线 d_{p-1} == 0
    if abs(d[p-1]) <= tol:
        d[p-1] = 0.0
        chase_zero_diagonal_vertical(q, p, d, e, V)
        return p - 1, True

    # 3. 扫描中间主对角线 d_k == 0
    for i in range(p - 2, q - 1, -1):
        if abs(d[i]) <= tol:
            d[i] = 0.0
            chase_zero_diagonal_horizontal(i, d, e, U, p)
            return p, False

    return p, False

In [11]:
import numpy as np

# ================= 1. Householder 双对角化 =================
def householder_bidiagonalization(A):
    m, n = A.shape
    B = A.copy().astype(float)
    U_vecs = []
    V_vecs = []

    for k in range(n):
        x = B[k:m, k]
        if len(x) > 1:
            e1 = np.zeros_like(x)
            e1[0] = 1.0
            sign_x = np.sign(x[0]) if x[0] != 0 else 1.0
            v_left = x + sign_x * np.linalg.norm(x) * e1
            norm_v_left = np.linalg.norm(v_left)
            if norm_v_left > 1e-15:
                v_left = v_left / norm_v_left
                U_vecs.append((k, v_left.copy()))
                B[k:m, k:n] -= 2.0 * np.outer(v_left, np.dot(v_left.T, B[k:m, k:n]))

        if k < n - 2:
            y = B[k, k+1:n]
            if len(y) > 1:
                e1_r = np.zeros_like(y)
                e1_r[0] = 1.0
                sign_y = np.sign(y[0]) if y[0] != 0 else 1.0
                v_right = y + sign_y * np.linalg.norm(y) * e1_r
                norm_v_right = np.linalg.norm(v_right)
                if norm_v_right > 1e-15:
                    v_right = v_right / norm_v_right
                    V_vecs.append((k, v_right.copy()))
                    B[k:m, k+1:n] -= 2.0 * np.outer(np.dot(B[k:m, k+1:n], v_right), v_right.T)

    B = np.triu(np.tril(B, 1))
    return B, U_vecs, V_vecs

# ================= 2. 显式重建 U 和 V 矩阵 =================
def build_UV_from_householder(m, n, U_vecs, V_vecs):
    U = np.eye(m)
    for k, v in reversed(U_vecs):
        U[k:m, :] -= 2.0 * np.outer(v, np.dot(v.T, U[k:m, :]))

    V = np.eye(n)
    for k, v in reversed(V_vecs):
        V[k+1:n, :] -= 2.0 * np.outer(v, np.dot(v.T, V[k+1:n, :]))

    return U, V

# ================= 3. SVD 的隐式 QR 迭代 (接入真实降阶) =================
def svd_bidiagonal_qr_iteration(B, U, V, num_iter=5000, tol=1e-10):
    m, n = U.shape[0], V.shape[0]
    d = np.diag(B).copy()
    e = np.diag(B, 1).copy()
    U_view = U[:, :n]

    p = n
    for i in range(num_iter):
        if p <= 1:
            print(f"SVD 隐式 QR 迭代 (追赶鼓包) 在第 {i+1} 次全部收敛。\n")
            break

        # 1. 物理降阶与断点扫描
        p_new, deflated = active_block_deflation(d, e, U_view, V, 0, p, tol)
        if deflated:
            p = p_new
            continue

        # 2. 找到当前活跃块 [q:p] 的左边界 q
        q = p - 1
        while q > 0 and e[q-1] != 0.0:
            q -= 1

        # 3. 仅对右下角的活跃子块进行鼓包追赶
        if q < p - 1:
            d_sub, e_sub = implicit_svd_qr_step(d[q:p], e[q:p-1], U_acc=U_view[:, q:p], V_acc=V[:, q:p])
            d[q:p] = d_sub
            e[q:p-1] = e_sub
    else:
        print("警告: 达到最大迭代次数，可能未完全收敛。")

    # 提取对角线作为奇异值，并确保奇异值为正
    sigma = d.copy()
    for k in range(n):
        if sigma[k] < 0:
            sigma[k] = -sigma[k]
            U[:, k] = -U[:, k]

    # 按奇异值降序排序
    sort_idx = np.argsort(sigma)[::-1]
    sigma = sigma[sort_idx]
    U[:, :n] = U[:, sort_idx]
    V[:, :] = V[:, sort_idx]

    return U, sigma, V

# ================= 4. 组装终极 SVD 算法与测试 =================
# np.random.seed(42)
n_A=500
m_A=300
A_orig = np.random.rand(n_A, m_A) * 10
print("原始矩阵 A (500x300) 部分展示:\n", A_orig[:3, :4])

B_bidiag, U_vecs, V_vecs = householder_bidiagonalization(A_orig)
U_init, V_init = build_UV_from_householder(n_A, m_A, U_vecs, V_vecs)

U_final, Sigma_final, V_final = svd_bidiagonal_qr_iteration(B_bidiag, U_init, V_init)

print("=== 自定义隐式 SVD 结果 ===")
print("奇异值 Sigma (前10个):\n", Sigma_final[:10])

Sigma_mat = np.zeros((n_A, m_A))
np.fill_diagonal(Sigma_mat, Sigma_final)
A_reconstructed = U_final @ Sigma_mat @ V_final.T

print("\n=== 正确性验证 ===")
print(f"重建 A 矩阵的最大误差: {np.max(np.abs(A_orig - A_reconstructed)):.4e}")

原始矩阵 A (500x300) 部分展示:
 [[0.6921 5.857  7.9887 7.6447]
 [6.7821 2.9886 0.4504 9.1467]
 [7.783  1.6047 5.653  5.0128]]
SVD 隐式 QR 迭代 (追赶鼓包) 在第 767 次全部收敛。

=== 自定义隐式 SVD 结果 ===
奇异值 Sigma (前10个):
 [1940.826   114.1717  112.9846  111.9279  110.9911  109.7806  108.537
  108.2694  107.2761  107.1395]

=== 正确性验证 ===
重建 A 矩阵的最大误差: 6.0756e-10


### 当我们只需要 Top-k 奇异值时：截断 SVD (Truncated SVD) 的主流算法

如果在实际应用中（如 PCA 降维、推荐系统、图像压缩等），我们不需要完整的 $O(n^3)$ 级别 SVD，而仅仅需要最大的 $k$ 个奇异值及其对应的奇异向量，这在数学上被称为**截断 SVD (Truncated SVD)**。

对于大矩阵求 Top-k 奇异值，直接计算全量分解是极大的浪费。目前工业界最主流的算法主要有以下两大家族：

#### 1. 随机化 SVD (Randomized SVD) —— 目前工业界的绝对霸主
这是近年来（尤其是 2011 年 Halko 等人发表奠基性论文后）在机器学习库中最火的算法。`scikit-learn` 中的 `TruncatedSVD` 和 `PCA` 默认在处理巨大矩阵时都会调用它。
*   **核心思想**：利用高斯随机矩阵乘法作为“探测器”，把原始巨大的矩阵 $A$ 投影到一个只比 $k$ 稍微大一点点的极小低维子空间中。在这个极小的子空间里，我们再使用传统的、高精度的标准 SVD（比如我们前面写的隐式 QR 迭代）。
*   **优点**：速度极其恐怖，天生适合 GPU 和分布式并行计算，对稠密大矩阵效果极好。
*   **时间复杂度**：直接从 $O(mn^2)$ 暴降到 $O(mnk)$。

#### 2. Lanczos / Arnoldi 迭代法 (Krylov 子空间法) —— 稀疏矩阵的黄金标准
如果你调用 `scipy.sparse.linalg.svds`，它的底层通常是由 Fortran 经典库 **ARPACK** 驱动的，这正是基于 Krylov 子空间方法的经典实现。
*   **核心思想**：我们前面手写了 Golub-Kahan 双对角化，如果矩阵极大，把它完整双对角化是不现实的。Lanczos 方法的核心是：并不需要显式给出矩阵 $A$，只需要能够计算 $Av$ 和 $A^T u$（矩阵-向量乘法）。它通过不断相乘构造一个 Krylov 子空间，在迭代 $k$ 次左右后，就能极其精准地逼近矩阵边缘（最大或最小）的几个奇异值。
*   **优点**：对于拥有极其庞大维度的**稀疏矩阵 (Sparse Matrix)**（例如 NLP 中的词频矩阵、图网络的邻接矩阵），它是最精确、最节省内存的绝对标准。

#### 3. 交替最小二乘法 (ALS) 与 幂法 (Power Iteration)
*   **幂法 (Power Iteration)**：如果 $k=1$（只求最大奇异值，如 PageRank 算法），随机初始化一个向量 $v$，不断计算 $v_{new} = A^T A v$ 就能收敛到最大特征向量。结合 Deflation（降阶，算出一个减去一个），也能勉强算 Top-k，但在相近特征值处容易有误差积累。
*   **ALS (Alternating Least Squares)**：在**推荐系统**（如协同过滤、隐语义模型）中极度流行。因为真实的评分矩阵往往极其稀疏且包含大量**缺失值**，无法直接用标准线性代数方法分解。ALS 把寻找 U 和 V 变成一个损失函数优化问题，固定 U 优化 V，再固定 V 优化 U，交替进行直到收敛。

---
**总结：技术选型路线**
- 矩阵极其稀疏，且对精度要求高 $\to$ **Lanczos 方法 (如 scipy.sparse.linalg.svds)**
- 矩阵是稠密的大数据，追求极限运算速度 $\to$ **Randomized SVD (如 sklearn.decomposition.TruncatedSVD)**
- 矩阵中包含大量缺失的空白值（推荐系统场景） $\to$ **ALS 交替最小二乘优化**

In [15]:
import numpy as np
import time
from sklearn.decomposition import TruncatedSVD
from scipy.sparse.linalg import svds

# ================= 1. 构造随机矩阵 =================
np.random.seed(42)
n_A, m_A = 2000, 1600
X = np.random.rand(n_A, m_A)

print(f"创建了随机稠密矩阵 X，维度: {X.shape}\n")

# ================= 2. 主流全量 SVD (LAPACK gesdd) =================
# 这是最经典的标准做法，底层高度优化，适合可以直接放入内存的中小型矩阵
start_time = time.time()
U_np, S_np, Vt_np = np.linalg.svd(X, full_matrices=False)
np_time = time.time() - start_time

# 计算重建误差
X_recon_np = U_np @ np.diag(S_np) @ Vt_np
max_err_np = np.max(np.abs(X - X_recon_np))

print(f"1. Numpy 全量 SVD (np.linalg.svd) 耗时: {np_time:.4f} 秒")
print(f"   最大奇异值: {S_np[0]:.4f}")
print(f"   重建矩阵最大误差: {max_err_np:.4e}")

# ================= 3. 随机化截断 SVD (Randomized SVD) =================
# 当矩阵极大，且我们只需要前 k 个主成分时，这是现代机器学习(如 PCA)的默认霸主
k = 10
start_time = time.time()
# algorithm='randomized' 是核心
rsvd = TruncatedSVD(n_components=k, algorithm='randomized', random_state=42)
rsvd.fit(X)
rsvd_time = time.time() - start_time

print(f"\n2. Sklearn 随机化截断 SVD (k={k}) 耗时: {rsvd_time:.4f} 秒")
print(f"   最大奇异值: {rsvd.singular_values_[0]:.4f}")

# ================= 4. Lanczos/Arnoldi 迭代法 (ARPACK) =================
# 虽然我们这里传入的是稠密矩阵，但它真正发威的地方是处理 scipy.sparse 的巨大稀疏矩阵
start_time = time.time()
# which='LM' 表示寻找 Largest Magnitude (绝对值最大) 的 k 个奇异值
U_sp, S_sp, Vt_sp = svds(X, k=k, which='LM')

# 注意：ARPACK 返回的奇异值默认是升序排列的，为了对比我们需要将其翻转为降序
S_sp = S_sp[::-1]
sp_time = time.time() - start_time

print(f"\n3. Scipy 稀疏截断 SVD (Lanczos/ARPACK) (k={k}) 耗时: {sp_time:.4f} 秒")
print(f"   最大奇异值: {S_sp[0]:.4f}")


创建了随机稠密矩阵 X，维度: (2000, 1600)

1. Numpy 全量 SVD (np.linalg.svd) 耗时: 6.8933 秒
   最大奇异值: 894.6912
   重建矩阵最大误差: 1.5832e-13

2. Sklearn 随机化截断 SVD (k=10) 耗时: 0.7723 秒
   最大奇异值: 894.6912

3. Scipy 稀疏截断 SVD (Lanczos/ARPACK) (k=10) 耗时: 1.7587 秒
   最大奇异值: 894.6912


### 终极揭秘：LAPACK 是如何做到如此之快的？ (Divide and Conquer 分治法)

在前面的代码中，我们亲手实现了 **隐式 QR 迭代 (Golub-Kahan 算法)**。在很长一段时间里（LAPACK 的 `gesvd` 函数），这就是世界上求 SVD 的最快方法。

但如果你仔细看刚才的测速，对于一个 500x300 的矩阵，`numpy.linalg.svd` 竟然只需要区区 0.05 秒！现代 LAPACK 默认调用的其实是另一个怪物级别的算法：**分治法 (Divide and Conquer, 对应 LAPACK 的 `gesdd` 函数)**。它比传统的 QR 迭代快了好几倍，甚至在某些大矩阵上能快一个数量级。（关于此算法，我决定单开一个ipynb文件学习）

分治法到底施了什么魔法？核心在于三步：**撕裂、分治、长期方程缝合**。

#### 1. 撕裂 (Tear)：用 Rank-1 更新切断矩阵
假设我们已经把原始矩阵变成了对称三对角矩阵 $T$（或者双对角阵 $B$）。
分治法极其暴力：它直接在矩阵正中间找到一条次对角线元素 $\beta$，然后把它强行写成一个对角块矩阵加上一个**秩为1的修正项 (Rank-1 Update)**：
$$ T = \begin{bmatrix} T_1 & 0 \\ 0 & T_2 \end{bmatrix} + \beta u u^T $$
其中 $T_1$ 和 $T_2$ 尺寸直接减半！$\beta u u^T$ 仅仅是一个由局部元素构成的极简矩阵。

#### 2. 分治 (Divide)：递归到底
既然切成了 $T_1$ 和 $T_2$，我们就递归地去求 $T_1$ 和 $T_2$ 的特征值分解。因为尺寸减半，计算量锐减。一直切，切到 1x1 或 2x2 的小矩阵，直接套公式秒解。

#### 3. 缝合 (Merge)：奇迹的长期方程 (Secular Equation)
难点在于：算出 $T_1$ 和 $T_2$ 的特征值后，怎么加上那个 $\beta u u^T$ 还原出原本大矩阵 $T$ 的特征值？
假设小矩阵的特征值组成对角阵 $D$，数学家推导出了一个极其优美的方程——**长期方程 (Secular Equation)**：
$$ f(\lambda) = 1 + \rho \sum_{i=1}^n \frac{z_i^2}{d_i - \lambda} = 0 $$
这里 $d_i$ 是小矩阵的特征值，$z_i$ 和 $\rho$ 都是已知的常数。要求原矩阵的特征值，就是求这个一维函数的根！

*   **降维打击**：本来需要进行复杂的矩阵迭代，现在变成了在一条曲线上求非线性方程的根（可以用极其快速的牛顿迭代法或有理插值法，平均两三步就能求出一个根）！
*   **完美的分离性质**：更神奇的是，$f(\lambda)$ 的根被完美地**夹在**相邻的两个 $d_i$ 之间（即 $d_i < \lambda_i < d_{i+1}$）。这意味着我们可以在不同的 CPU 核心上**完全独立、无锁地并行求解**每一个特征值！

#### 4. 极致压榨硬件：BLAS Level 3 的胜利
QR 迭代（追赶鼓包）为什么慢？因为 Givens 旋转每次只能修改相邻的**两行或两列**。这种操作在 BLAS (基础线性代数子程序) 中属于 **Level 1 (向量-向量)** 或 **Level 2 (矩阵-向量)** 操作。CPU 需要不断从内存里把数据倒腾到缓存里，极其浪费带宽 (Memory-Bound)。

而分治法在求出特征值后，需要用矩阵乘法把正交矩阵拼起来。这属于 **BLAS Level 3 (矩阵乘矩阵)** 操作！这种操作可以完美切块 (Block)，极其契合现代 CPU 的多级缓存 (L1/L2/L3 Cache) 和 SIMD 矢量指令集 (如 AVX-512)。计算单元 (Compute-Bound) 被 100% 跑满，没有任何性能浪费。

**总结**：
LAPACK `gesdd` 惊人的速度，来源于它将“矩阵运算”巧妙地转化为了“标量方程求根”，并把零碎的列变换整合成了能把 CPU 缓存跑出火星的“块矩阵乘法”。这是现代算法设计与底层计算机体系结构完美融合的最高杰作！

### 拓展：张量网络 (Tensor Networks) 中的截断 SVD 是哪种情况？

在量子多体物理和前沿机器学习中，**张量网络（如 MPS, DMRG, PEPS, MERA）**对 SVD 的依赖到了极其疯狂的程度。每次相邻张量发生收缩（Contraction）后，维度都会爆炸，必须立刻通过截断 SVD 将“键维 (Bond Dimension, $\chi$)”压缩回可控范围内。

在张量网络中，我们要面对的矩阵是通过将高维张量“拉平 (Reshape / Matricize)”得到的。它们的具体情况和技术选型如下：

#### 1. 矩阵的基本特征：(分块) 稠密矩阵
大部分张量重构出来的矩阵是**完全稠密 (Dense)** 的。如果物理系统引入了对称性（如 $U(1)$ 粒子数守恒或 $SU(2)$ 自旋对称性），矩阵会变成**分块对角稠密矩阵 (Block-diagonal Dense Matrix)**。这与图计算或NLP中那种极度稀疏的矩阵有着本质区别。

#### 2. 张量网络中截断 SVD 的三大主力算法
根据截断场景的不同，张量网络领域会采用不同的算法：

*   **情景 A：中小规模的显式张量截断 $\to$ 直接硬算全量 SVD (LAPACK `gesdd` 分治法)**
    *   **情况**：在一维张量网络 (MPS) 中，键维 $\chi$ 通常在 $100 \sim 2000$ 之间。虽然叫“截断”，但由于矩阵维度不够大，底层的 BLAS/LAPACK 库（基于 Divide and Conquer 算法）优化得太好了。
    *   **做法**：哪怕我们只需要 Top-100 的奇异值，工业界通常直接把几千维的稠密矩阵进行**全量 SVD**，然后手动扔掉后面的奇异值。这在 CPU 上往往比复杂的迭代法还要快，而且数值稳定性最高。

*   **情景 B：极大规模的显式张量截断 (如 PEPS 演化) $\to$ 随机化 SVD (Randomized SVD)**
    *   **情况**：在二维张量网络 (PEPS) 或使用极大键维时，拉平后的稠密矩阵可能达到 $10^4 \times 10^4$ 甚至更大，而我们需要截断保留的 $k$ 远小于矩阵维度。此时全量 SVD 会导致 OOM 或极其缓慢。
    *   **做法**：**随机化 SVD 是现代张量网络库（如 ITensor, Google TensorNetwork）在处理大稠密张量时的首选**。因为张量操作非常容易并行化，RSVD 中密集的“矩阵-随机矩阵”乘法能将 GPU 的并行算力压榨到极限。

*   **情景 C：隐式矩阵的极端截断 / 求基态 $\to$ Lanczos / Davidson 迭代法**
    *   **情况**：在著名的 DMRG 算法中寻找局域基态时，我们要对“有效哈密顿量 (Effective Hamiltonian)”求绝对值最大（或能量最低）的几个特征值。这个环境哈密顿量张量极其巨大（可能是 $10^6 \times 10^6$），**根本无法在内存中显式写成一个矩阵**。
    *   **做法**：使用基于 Krylov 子空间的 **Lanczos** 算法，或者专门针对物理多体系统的 **Davidson** 算法。此时我们不需要写出矩阵，只需要定义如何将这个隐形的哈密顿量与一个试探张量进行**收缩 (Tensor Contraction)**，这完美契合了 Lanczos 只需要“矩阵乘向量 $Av$”的特性。

---
**总结**：
张量网络的核心是**稠密张量收缩**。如果你在写普通的 MPS，直接调底层的标准 SVD 往往最省事；如果你在上高维或者 GPU，果断切换到**随机化 SVD**；如果是求解 DMRG 局域基态这种无法写出矩阵的隐式算子，则必须求助于 **Lanczos/Davidson**。